# Trajectory and velocity analysis of scRNAseq COLON data 

This notebook will guide your through the analysis... This is an example for the **HEALTHY** dataset.

## 0. Imports and settings

In [ ]:
import scanpy as sc
import scvelo as scv
import numpy as np
import pandas as pd
import mnnpy
import matplotlib.pyplot as plt
import seaborn as sb
import scanpy.api as scapi
import scrublet as scr
import doubletdetection

# Set some decent font (probably will have to change to Helvetica for publication)
from matplotlib import rcParams
rcParams['font.family'] = ['Georgia serif']

# Box for text background in PAGA
bbox = dict(bbox=dict(boxstyle="round", ec='white', fc="white", alpha=0.5, linewidth=0))

# List of genes associated with cell cycle.
cell_cycle_genes = [x.strip() for x in open('data/cell_cycle_genes.txt')]
cell_cycle_genes = cell_cycle_genes[1:]
s_genes = cell_cycle_genes[:43]
g2m_genes = cell_cycle_genes[43:]

# Proliferation markers
ProlifMarkers = ['RAD51C','SLC7A2','CCDC18','DCTD','RAD54B','BARD1','KLHL23','FIGNL1','POLA1','DEPDC1','PPIL5','PPAT','C6ORF150','XRCC2','SKA1','SLC12A2','RCC2','KIF18A','KIF11','ESCO2','E2F7','RAD54L','CHEK1','PRKD3','NA','CLSPN','KIAA0101','DTL','SLC7A2','ATAD5','POLE','FANCB','CENPA','NEXN','PPIL5','FAM111A','TTPA','CDC7','NAP1L1','HEMGN','KNTC1','PRKD3','TBC1D19','SKA3','NCAPG2','POLE2','EXO1','CENPI','SGOL2','CENPN','DTL','CENPN','NEIL3','EXO1','HMMR','RAD54L','BUB1','MCM3','NRM','CNN3','ALMS1','TIMELESS','ATAD2','RRP1B','AURKB','SLC16A12','RAD51C','MCM3','TPX2','C1ORF135','KIFC1','TBC1D4','CHEK1','ZWILCH','SCML2','FADS1','GINS2','MYC','TIMELESS','AQP4','PCK2','CDC45L','ANKRD26','PPAT','HELLS','C15ORF42','ZNF473','CDCA2','NA','HMMR','UBE2T','BRCA1','KIF11','KIF20B','PLK4','PRR11','TMEFF1','CNN3','NA','MCM5','MCM2','GINS1','TMEM107','ERCC6L','CASC5','IFITM3','RAD51AP1','RAD51AP1','BUB1','NCAPH','CBX6','DTL','BLM','GINS2','PRR11','9430037O13RIK','MORC4','PFAS','FGR','RAD54B','C6ORF173','NCAPH','NA','PALB2','MELK','NSL1','CDCA2','NAP1L1','C6ORF150','MBD4','MCM3','CENPF','MCM7','RACGAP1','TIPIN','MYB','C14ORF106','C16ORF48','RAD18','CENPE','RAD51AP1','CHEK1','MCM3','UHRF1','CEP55','FOXM1','SLFN13','IQGAP3','TUBE1','NUF2','CCDC99','RAD51','FAM72A','CDCA5','SMC2','KIF18B','MCM3','RACGAP1','GLS2','DUT','TMEM107','BRCA1','AU016916','MYB','2700099C18RIK','GTSE1','RAD54L','HELLS','WDR35','PLK1','TIMP3','RPA2','CEP55','NA','RAD18','C6ORF167','C4ORF21','PRIM2','C12ORF48','BUB1B','C1ORF112','NA','LAMC1','ANLN','TRIM37','RRM2','CCNA2','HAUS5','ESPL1','TCF19','FOXM1','DDX11','KIF2C','CCNE1','PUS7','LPHN1','KBTBD6','NA','TXNDC16','DACH1','MASTL','TRIM37','ZRANB3','SIVA1','2810454L23RIK','CELSR2','PSIP1','MCM8','SHCBP1','ASF1B','KCNN4','AU020206','PRIM1','CKS1B','CHAF1B','MCM2','RCC2','RASA3','DUT','CTPS','BAG2','SIVA1','CKLF','PRIM2','DUS4L','TMEM194B','TUBB','HAUS6','DCAF12L1','MARCKS','SETDB1','2810408B13RIK','SYCE2','UNG','NA','SEMA3C','CLCA4','MLF1IP','PAICS','MCM7','POLD1','C17ORF53','E2F1','ZC3H7B','TRAIP','LOC388559','MPHOSPH9','WDHD1','BCL7A','NA','ATIC','CCBE1','TSGA14','HELLS','MASTL','SPAG5','LYSMD2','C15ORF23','CEP76','DNAJC18','TAF4B','CDC6','SMC2','CENPE','4930513N10RIK','LYAR','CDK2','FOXM1','CKLF','NCAPD2','SLFN13','RPUSD2','VRK1','CYP39A1','DLGAP5','INCENP','PBK','TMEFF1','NUSAP1','NEK2','C3ORF26','VRK1','PSMC3IP','POLI','NFATC2','D17H6S56E-5','PGM2L1','AURKB','CEP97','CKAP2L','FANCB','DNMT1','CCNF','BRCA1','ASPM','HMMR','HMGA2','DSN1','NA','CDCA5','PGM2L1','IGF1R','ZNF367','WDR34','NUP210','EXOSC8','GEN1','LIG1','TPX2','HMMR','TK1','PLK4','TRIM68','HAUS6','FAM54A','C8ORF79','AEN','ZNRF3','PSMC3IP','CCDC14','DNA2','CENPP','C21ORF91','DLGAP5','DCK','SLC7A5','HAUS4','CHTF18','SEMA4D','PGM2L1','ZBTB25','UHRF1','TSGA14','BCKDHB','MCM4','KIF22','CHEK1','NAP1L1','RRM1','FTSJD1','SGOL1','MDN1','FEN1','RRM2','CKLF','NEK2','QSOX2','RRP15','NCAPG','AURKA','MYBL2','NA','PAICS','PIP4K2B','SPC24','DNA2','PRC1','RPP40','KIF24','MPHOSPH9','TMEM173','TMEM48','NIN','WDR76','ZNF275','UTP15','3110040M04RIK','MTHFD2','TACC3','NA','RBBP6','CENPF','ZMYND19','SKA2','MTBP','NDE1','MYBL2','EXOSC2','DIAPH3','CDC2','CP110','KIAA0649','GEMIN5','DUSP7','POLR1E','NUP133','TRIM37','TSPAN12','NAP1L1','PRC1','GEMIN4','GINS3','TOPBP1','NEUROG3','MPHOSPH9','CASP12','WWTR1','NAP1L1','MCM7','RFC3','ZMYND19','KIF4A','LMNB1','RFC3','BBS7','ILF3','ZNF704','GSTCD','PRIM1','NA','ZBTB16','NA','DKC1','QTRT1','TRIP13','SLC1A5','CDCA7L','C14ORF143','MCM5','TRIP13','2810442I21RIK','DNAH11','4933439C10RIK','POLD1','CKLF','CEP192','MIRHG1','ZFP783','NUP85','C3ORF26','TLR2','C6ORF167']
ProlifMarkers = list(set(ProlifMarkers))

## 1. Load data

All files are in the `.loom` format. They have been preprcoessed with *cellranger* (demultiplexing, alignment and counting) and *velocyto* (annotation of spliced/unspliced counts). 

After loading in the file, we have to remove some of the unnecssary data fields generated by previous processing. What we want in the end is just the matrix of raw counts, and matrices of spliced and unspliced reads.

In [ ]:
sample_name = "healthy0"
file = "data/{}.loom".format(sample_name)
adata0 = scv.read_loom(file, sparse=True, cleanup=True)
adata0.var_names_make_unique()
# Delete unnecessary data
del adata0.obs['Clusters']
del adata0.obs['_X']
del adata0.obs['_Y']
del adata0.var['Accession']
del adata0.var['Chromosome']
del adata0.var['End']
del adata0.var['Start']
del adata0.var['Strand']
del adata0.layers['ambiguous']

sample_name = "healthy1"
file = "data/{}.loom".format(sample_name)
adata1 = scv.read_loom(file, sparse=True, cleanup=True)
adata1.var_names_make_unique()
del adata1.obs['Clusters']
del adata1.obs['_X']
del adata1.obs['_Y']
del adata1.var['Accession']
del adata1.var['Chromosome']
del adata1.var['End']
del adata1.var['Start']
del adata1.var['Strand']
del adata1.layers['ambiguous']

sample_name = "healthy7"
file = "data/{}.loom".format(sample_name)
adata7 = scv.read_loom(file, sparse=True, cleanup=True)
adata7.var_names_make_unique()
del adata7.obs['Clusters']
del adata7.obs['_X']
del adata7.obs['_Y']
del adata7.var['Accession']
del adata7.var['Chromosome']
del adata7.var['End']
del adata7.var['Start']
del adata7.var['Strand']
del adata7.layers['ambiguous']

sample_name = "healthy8"
file = "data/{}.loom".format(sample_name)
adata8 = scv.read_loom(file, sparse=True, cleanup=True)
adata8.var_names_make_unique()
del adata8.obs['Clusters']
del adata8.obs['_X']
del adata8.obs['_Y']
del adata8.var['Accession']
del adata8.var['Chromosome']
del adata8.var['End']
del adata8.var['Start']
del adata8.var['Strand']
del adata8.layers['ambiguous']

## 2. Quality control

Before we can analyse the samples properly we need to filter the cells and genes based on their quality. We start by joining the samples together and calculating some standard quality metrics. 

In [ ]:
# Concatenate the files to calculate joint QC metrics
adata_qc = adata0.concatenate(adata1, adata7, adata8, 
                              batch_key='Sample', batch_categories=['healthy0', 'healthy1', 'healthy7', 'healthy8'])

# Calculate QC covariates
adata_qc.obs['n_counts'] = adata_qc.X.sum(1)
adata_qc.obs['n_spliced'] = adata_qc.layers['spliced'].sum(1)
adata_qc.obs['n_unspliced'] = adata_qc.layers['unspliced'].sum(1)
adata_qc.obs['n_genes'] = (adata_qc.X > 0).sum(1)
# Compute for each cell the fraction of counts in mito genes vs. all genes
mito_genes = adata_qc.var_names.str.startswith('MT-')
adata_qc.obs['percent_mito'] = np.sum(
    adata_qc[:, mito_genes].X, axis=1) / np.sum(adata_qc.X, axis=1)
ribo_genes = adata_qc.var_names.str.startswith('RP')
adata_qc.obs['percent_ribo'] = np.sum(
    adata_qc[:, ribo_genes].X, axis=1) / np.sum(adata_qc.X, axis=1)
adata_qc.obs['percent_MALAT1'] = np.sum(
    adata_qc[:, 'MALAT1'].X, axis=1) / np.sum(adata_qc.X, axis=1)

# Initial plotting settings for more detailed scatter and dist plots.
#scv.settings.set_figure_params('scvelo', dpi=150, vector_friendly=False)

sc.pl.violin(adata_qc, ['n_genes', 'n_counts'], jitter=0.4, groupby='Sample')
sc.pl.violin(adata_qc, ['percent_mito', 'percent_ribo'], jitter=0.4, groupby='Sample')
sc.pl.violin(adata_qc, ['n_spliced', 'n_unspliced'], jitter=0.4, groupby='Sample')

As can be seen in the figure above, there are significant differences in the distributions of quality metrics in the four samples. This is why filtering based on those metrics is done separately for each of the samples.

### 2.1 healthy0 QC 

In [ ]:
# Calculate QC covariates
adata0.obs['n_counts'] = adata0.X.sum(1)
adata0.obs['log_counts'] = np.log(adata0.obs['n_counts'])
adata0.obs['n_spliced'] = adata0.layers['spliced'].sum(1)
adata0.obs['n_unspliced'] = adata0.layers['unspliced'].sum(1)
adata0.obs['n_genes'] = (adata0.X > 0).sum(1)
# Compute for each cell the fraction of counts in mito genes vs. all genes
mito_genes = adata0.var_names.str.startswith('MT-')
adata0.obs['percent_mito'] = np.sum(
    adata0[:, mito_genes].X, axis=1) / np.sum(adata0.X, axis=1)
# Compute for each cell the fraction of counts in ribosomal genes vs. all genes
ribo_genes = adata0.var_names.str.startswith('RP')
adata0.obs['percent_ribo'] = np.sum(
    adata0[:, ribo_genes].X, axis=1) / np.sum(adata0.X, axis=1)

sc.set_figure_params(fontsize=12, vector_friendly=True, dpi=150, dpi_save=300)
sc.pl.scatter(adata0, 'n_counts', 'n_genes', color='percent_mito', title='Percent mitochondrial (colour)', save='qc_healhty')
sc.pl.scatter(adata0, 'n_spliced', 'n_unspliced', save='spliced_healhty')

There is a strong correlation between the number of counts, genes and the mitochondrial content, however, it is still important to look at those statistics jointly. There are many outliers (high number of counts, low number of genes). Specifically, we can see cells with many genes (likely not dead) that have a high mitochondrial content (around 50%). This means we cannot simply exclude all the cells with more than 5% (as done in other studies). 

Here we use the tooll called scrublet to identify doublets - two cells barcoded togheter. Especially important when we are trying too identify cells transitioning between celltypes. We use scrublet to assign doublet score (probability of being a doublet). Actual doublets are detected with the Doublet Detection tool, which is more sofisticated. Uses a classsifier and does not need us to supply a cutoff value for being a doublet. 

In [ ]:
# Doublet analysis
scrub = scr.Scrublet(adata0.X, expected_doublet_rate=0.06)
doublet_scores, predicted_doublets = scrub.scrub_doublets(min_counts=2, 
                                                          min_cells=3, 
                                                          min_gene_variability_pctl=85, 
                                                          n_prin_comps=30,
                                                          mean_center=False, 
                                                          normalize_variance=False,
                                                          log_transform=True)

In [ ]:
scrub.plot_histogram()

In [ ]:
scrub.set_embedding('UMAP', scr.get_umap(scrub.manifold_obs_, 10, min_dist=0.3))
scrub.plot_embedding('UMAP', order_points=True)

In [ ]:
adata0.obs['doublet_score'] = scrub.doublet_scores_obs_

Another tool to annotate doublets is Doublet Detection. More sophisticated, we use it to actually call doublets.

In [ ]:
clf = doubletdetection.BoostClassifier(n_iters=50, use_phenograph=False, standard_scaling=True)
doublets = clf.fit(adata0.X).predict(p_thresh=1e-16, voter_thresh=0.5)

In [ ]:
f = doubletdetection.plot.convergence(clf, show=True, p_thresh=1e-16, voter_thresh=0.5)

In [ ]:
f2, umap_coords = doubletdetection.plot.umap_plot(adata0.X, doublets, random_state=1, show=True)

In [ ]:
adata0.obs['doublet'] = list(map(str, map(int, doublets)))

We look at distribution plots of **count** statistics to set tresholds for **cell filtering**. 

In [ ]:
#Thresholding decision: spliced counts
ax=sb.distplot(adata0.obs['n_spliced'], kde=False)
plt.show()

ax=sb.distplot(adata0.obs['n_spliced'][adata0.obs['n_spliced']<3500], kde=False, bins=60)
plt.show()

ax=sb.distplot(adata0.obs['n_spliced'][adata0.obs['n_spliced']>15000], kde=False, bins=60)
plt.show()

Let's look at unspliced counts.

In [ ]:
#Thresholding decision: unspliced counts
ax=sb.distplot(adata0.obs['n_unspliced'], kde=False)
plt.show()

ax=sb.distplot(adata0.obs['n_unspliced'][adata0.obs['n_unspliced']<2000], kde=False, bins=60)
plt.show()

We look at distribution plots of **gene** statistics to set tresholds for **cell filtering**. 

In [ ]:
#Thresholding decision: genes
ax=sb.distplot(adata0.obs['n_genes'], kde=False, bins=60)
plt.show()
ax=sb.distplot(adata0.obs['n_genes'][adata0.obs['n_genes']<1000], kde=False, bins=60)
plt.show()

By visually tracing a Gaussian around the population we estimate the treshloding limits.

In [ ]:
print("Before filtering cells:\n", adata0, "\n")

sc.pp.filter_cells(adata0, min_counts=1250)
sc.pp.filter_cells(adata0, max_counts=18750)
adata0 = adata0[adata0.obs['n_unspliced']>600]
sc.pp.filter_cells(adata0, min_genes=250)

print("After filtering cells:\n", adata0)

The distribution of genes and counts is now much more balanced.

In [ ]:
sc.pl.violin(adata0, ['n_genes', 'n_counts', 'percent_mito'],
             jitter=0.4, multi_panel=True)

Let's now focus on the the **mitochondrial content**.

In [ ]:
sc.pl.scatter(adata0, 'n_genes', 'percent_mito')

Now we look at distribution of the mitochondrial content and compare it with the number of genes. In general, the content is quite high across all the cells in the sample and since we want to capture the majority of the population, **we set the limit to 50%**. That is a very significant amount, however, those cells still have a lot of genes, which suggests that they are alive and biologically relevant. Another explanation could be that the high content is due to a technical effect and affects all the cells, meaning it does not accurately represents variation in our dataset. 

In any case, we should be permissive with our treshold.

In [ ]:
adata0 = adata0[adata0.obs['percent_mito'] < 0.3, :]
sc.pl.scatter(adata0, 'n_counts', 'n_genes', color='percent_mito', title='Percent mitochondrial genes')
print(adata0)

Next we filter out the remaining doublet cells. We filter based on the classification from the Doublet Detection method. We compare the agreement between the methods by then visualising the score from the scrublet method. We can see that some cells that scored very highly in scrublet are not classified as doublets by the other software. Those may be false negatives, so we filter them out as well to be sure we don't have any doublets. This is done at the cost of filtering out some false positives. However, the assumption is that the cell populations we are interested in are common enough that they would not be affected. 

In [ ]:
sc.pl.scatter(adata0, 'n_genes', 'doublet_score', color='doublet', palette=['gray', 'orange'], legend_loc='none', title='', save='healthy_doublet')

In [ ]:
adata0 = adata0[adata0.obs['doublet'] != '1']
adata0 = adata0[adata0.obs['doublet_score'] < 0.3]

Finally, we can filter out genes. This is done mostly to decrease the size of the object and speed up calculations later on by getting rid off uninformative genes. We do it per sample, so that we remove sample specific genes later on. We keep genes that have counts in at least 5 cells. Filtering out genes also helps speed up subsequent calculations.

In [ ]:
sc.pl.highest_expr_genes(adata0, n_top=20)
print(adata0)

### 2.2 healthy1 QC

The analysis and its reasoning is analogous to the previous one. See section **2.1** for describtion.

In [ ]:
# Calculate QC covariates
adata1.obs['n_counts'] = adata1.X.sum(1)
adata1.obs['log_counts'] = np.log(adata1.obs['n_counts'])
adata1.obs['n_genes'] = (adata1.X > 0).sum(1)
adata1.obs['n_spliced'] = adata1.layers['spliced'].sum(1)
adata1.obs['n_unspliced'] = adata1.layers['unspliced'].sum(1)
# Compute for each cell the fraction of counts in mito genes vs. all genes
mito_genes = adata1.var_names.str.startswith('MT-')
adata1.obs['percent_mito'] = np.sum(
    adata1[:, mito_genes].X, axis=1) / np.sum(adata1.X, axis=1)
# Compute for each cell the fraction of counts in ribosomal genes vs. all genes
ribo_genes = adata1.var_names.str.startswith('RP')
adata1.obs['percent_ribo'] = np.sum(
    adata1[:, ribo_genes].X, axis=1) / np.sum(adata1.X, axis=1)

sc.pl.scatter(adata1, 'n_counts', 'n_genes', color='percent_mito', title='Percent mitochondrial genes')
sc.pl.scatter(adata1, 'n_spliced', 'n_unspliced')

In [ ]:
# Doublet analysis
scrub = scr.Scrublet(adata1.X, expected_doublet_rate=0.06)
doublet_scores, predicted_doublets = scrub.scrub_doublets(min_counts=2, 
                                                          min_cells=3, 
                                                          min_gene_variability_pctl=85, 
                                                          n_prin_comps=30,
                                                          mean_center=False, 
                                                          normalize_variance=False,
                                                          log_transform=True)

In [ ]:
scrub.plot_histogram()

In [ ]:
scrub.set_embedding('UMAP', scr.get_umap(scrub.manifold_obs_, 10, min_dist=0.3))
scrub.plot_embedding('UMAP', order_points=True)

In [ ]:
adata1.obs['doublet_score'] = scrub.doublet_scores_obs_

In [ ]:
clf = doubletdetection.BoostClassifier(n_iters=50, use_phenograph=False, standard_scaling=True)
doublets = clf.fit(adata1.X).predict(p_thresh=1e-16, voter_thresh=0.5)

In [ ]:
f = doubletdetection.plot.convergence(clf, show=True, p_thresh=1e-16, voter_thresh=0.5)

In [ ]:
f2, umap_coords = doubletdetection.plot.umap_plot(adata1.X, doublets, random_state=1, show=True)

In [ ]:
adata1.obs['doublet'] = list(map(str, map(int, doublets)))

In [ ]:
#Thresholding decision: spliced counts
ax=sb.distplot(adata1.obs['n_spliced'], kde=False)
plt.show()

ax=sb.distplot(adata1.obs['n_spliced'][adata1.obs['n_spliced']<4000], kde=False, bins=60)
plt.show()

ax=sb.distplot(adata1.obs['n_spliced'][adata1.obs['n_spliced']>20000], kde=False, bins=60)
plt.show()

Let's look at unspliced counts.

In [ ]:
#Thresholding decision: unspliced counts
ax=sb.distplot(adata1.obs['n_unspliced'], kde=False)
plt.show()

ax=sb.distplot(adata1.obs['n_unspliced'][adata1.obs['n_unspliced']<1750], kde=False, bins=60)
plt.show()

In [ ]:
#Thresholding decision: genes
ax=sb.distplot(adata1.obs['n_genes'], kde=False, bins=60)
plt.show()
ax=sb.distplot(adata1.obs['n_genes'][adata1.obs['n_genes']<1750], kde=False, bins=60)
plt.show()

In [ ]:
print("Before filtering cells:\n", adata1, "\n")

sc.pp.filter_cells(adata1, min_counts=2100)
sc.pp.filter_cells(adata1, max_counts=26000)
adata1 = adata1[adata1.obs['n_unspliced']>700]
sc.pp.filter_cells(adata1, min_genes=900)

print("After filtering cells:\n", adata1)

In [ ]:
sc.pl.violin(adata1, ['n_genes', 'n_counts', 'percent_mito'],
             jitter=0.4, multi_panel=True)

In [ ]:
sc.pl.scatter(adata1, 'n_genes', 'percent_mito')

In [ ]:
adata1 = adata1[adata1.obs['percent_mito'] < 0.3, :]
sc.pl.scatter(adata1, 'n_counts', 'n_genes', color='percent_mito', title='Percent mitochondrial genes')
print(adata1)

In [ ]:
sc.pl.scatter(adata1, 'n_genes', 'doublet_score', color='doublet', palette=['gray', 'orange'], legend_loc='none', title='Doublet')
adata1 = adata1[adata1.obs['doublet'] != '1']
adata1 = adata1[adata1.obs['doublet_score'] < 0.3]

In [ ]:
sc.pl.highest_expr_genes(adata1, n_top=20)
print(adata1)

### 2.3 healthy7 QC

The analysis and its reasoning is analogous to the previous one. See section **2.1** for describtion.

In [ ]:
# Calculate QC covariates
adata7.obs['n_counts'] = adata7.X.sum(1)
adata7.obs['log_counts'] = np.log(adata7.obs['n_counts'])
adata7.obs['n_genes'] = (adata7.X > 0).sum(1)
adata7.obs['n_spliced'] = adata7.layers['spliced'].sum(1)
adata7.obs['n_unspliced'] = adata7.layers['unspliced'].sum(1)
# Compute for each cell the fraction of counts in mito genes vs. all genes
mito_genes = adata7.var_names.str.startswith('MT-')
adata7.obs['percent_mito'] = np.sum(
    adata7[:, mito_genes].X, axis=1) / np.sum(adata7.X, axis=1)
# Compute for each cell the fraction of counts in ribosomal genes vs. all genes
ribo_genes = adata7.var_names.str.startswith('RP')
adata7.obs['percent_ribo'] = np.sum(
    adata7[:, ribo_genes].X, axis=1) / np.sum(adata7.X, axis=1)

sc.pl.scatter(adata7, 'n_counts', 'n_genes', color='percent_mito', title='Percent mitochondrial genes')
sc.pl.scatter(adata7, 'n_spliced', 'n_unspliced')

In [ ]:
# Doublet analysis
scrub = scr.Scrublet(adata7.X, expected_doublet_rate=0.06)
doublet_scores, predicted_doublets = scrub.scrub_doublets(min_counts=2, 
                                                          min_cells=3, 
                                                          min_gene_variability_pctl=85, 
                                                          n_prin_comps=30,
                                                          mean_center=False, 
                                                          normalize_variance=False,
                                                          log_transform=True)

In [ ]:
scrub.plot_histogram()

In [ ]:
scrub.set_embedding('UMAP', scr.get_umap(scrub.manifold_obs_, 10, min_dist=0.3))
scrub.plot_embedding('UMAP', order_points=True)

In [ ]:
adata7.obs['doublet_score'] = scrub.doublet_scores_obs_

In [ ]:
clf = doubletdetection.BoostClassifier(n_iters=50, use_phenograph=False, standard_scaling=True)
doublets = clf.fit(adata7.X).predict(p_thresh=1e-16, voter_thresh=0.5)

In [ ]:
f = doubletdetection.plot.convergence(clf, show=True, p_thresh=1e-16, voter_thresh=0.5)

In [ ]:
f2, umap_coords = doubletdetection.plot.umap_plot(adata7.X, doublets, random_state=1, show=True)

In [ ]:
adata7.obs['doublet'] = list(map(str, map(int, doublets)))

In [ ]:
#Thresholding decision: spliced counts
ax=sb.distplot(adata7.obs['n_spliced'], kde=False)
plt.show()

ax=sb.distplot(adata7.obs['n_spliced'][adata7.obs['n_spliced']<2500], kde=False, bins=60)
plt.show()

ax=sb.distplot(adata7.obs['n_spliced'][adata7.obs['n_spliced']>12500], kde=False, bins=60)
plt.show()

Let's look at unspliced counts.

In [ ]:
#Thresholding decision: unspliced counts
ax=sb.distplot(adata7.obs['n_unspliced'], kde=False)
plt.show()

ax=sb.distplot(adata7.obs['n_unspliced'][adata7.obs['n_unspliced']<1500], kde=False, bins=60)
plt.show()

In [ ]:
#Thresholding decision: genes
ax=sb.distplot(adata7.obs['n_genes'], kde=False, bins=60)
plt.show()
ax=sb.distplot(adata7.obs['n_genes'][adata7.obs['n_genes']<1000], kde=False, bins=60)
plt.show()

In [ ]:
print("Before filtering cells:\n", adata7, "\n")

sc.pp.filter_cells(adata7, min_counts=1600)
sc.pp.filter_cells(adata7, max_counts=18000)
adata7 = adata7[adata7.obs['n_unspliced']>750]
sc.pp.filter_cells(adata7, min_genes=500)

print("After filtering cells:\n", adata7)

In [ ]:
sc.pl.violin(adata7, ['n_genes', 'n_counts', 'percent_mito'],
             jitter=0.4, multi_panel=True)

In [ ]:
sc.pl.scatter(adata7, 'n_genes', 'percent_mito')

In [ ]:
adata7 = adata7[adata7.obs['percent_mito'] < 0.3, :]
sc.pl.scatter(adata7, 'n_counts', 'n_genes', color='percent_mito', title='Percent mitochondrial genes)')
print(adata7)

In [ ]:
sc.pl.scatter(adata7, 'n_genes', 'doublet_score', color='doublet', palette=['gray', 'orange'], legend_loc='none', title='Doublet')
adata7 = adata7[adata7.obs['doublet'] != '1']
adata7 = adata7[adata7.obs['doublet_score'] < 0.3]

In [ ]:
sc.pl.highest_expr_genes(adata7, n_top=20)
print(adata7)

### 2.4 healthy8 QC

The analysis and its reasoning is analogous to the previous one. See section **2.1** for describtion.

In [ ]:
# Calculate QC covariates
adata8.obs['n_counts'] = adata8.X.sum(1)
adata8.obs['log_counts'] = np.log(adata8.obs['n_counts'])
adata8.obs['n_genes'] = (adata8.X > 0).sum(1)
adata8.obs['n_spliced'] = adata8.layers['spliced'].sum(1)
adata8.obs['n_unspliced'] = adata8.layers['unspliced'].sum(1)
# Compute for each cell the fraction of counts in mito genes vs. all genes
mito_genes = adata8.var_names.str.startswith('MT-')
adata8.obs['percent_mito'] = np.sum(
    adata8[:, mito_genes].X, axis=1) / np.sum(adata8.X, axis=1)
# Compute for each cell the fraction of counts in ribosomal genes vs. all genes
ribo_genes = adata8.var_names.str.startswith('RP')
adata8.obs['percent_ribo'] = np.sum(
    adata8[:, ribo_genes].X, axis=1) / np.sum(adata8.X, axis=1)

sc.pl.scatter(adata8, 'n_counts', 'n_genes', color='percent_mito', title='Percent mitochondrial genes')
sc.pl.scatter(adata8, 'n_spliced', 'n_unspliced')

In [ ]:
# Doublet analysis
scrub = scr.Scrublet(adata8.X, expected_doublet_rate=0.06)
doublet_scores, predicted_doublets = scrub.scrub_doublets(min_counts=2, 
                                                          min_cells=3, 
                                                          min_gene_variability_pctl=85, 
                                                          n_prin_comps=30,
                                                          mean_center=False, 
                                                          normalize_variance=False,
                                                          log_transform=True)

In [ ]:
scrub.plot_histogram()

In [ ]:
scrub.set_embedding('UMAP', scr.get_umap(scrub.manifold_obs_, 10, min_dist=0.3))
scrub.plot_embedding('UMAP', order_points=True)

In [ ]:
adata8.obs['doublet_score'] = scrub.doublet_scores_obs_

In [ ]:
clf = doubletdetection.BoostClassifier(n_iters=50, use_phenograph=False, standard_scaling=True)
doublets = clf.fit(adata8.X).predict(p_thresh=1e-16, voter_thresh=0.5)

In [ ]:
f = doubletdetection.plot.convergence(clf, show=True, p_thresh=1e-16, voter_thresh=0.5)

In [ ]:
f2, umap_coords = doubletdetection.plot.umap_plot(adata8.X, doublets, random_state=1, show=True)

In [ ]:
adata8.obs['doublet'] = list(map(str, map(int, doublets)))

In [ ]:
#Thresholding decision: spliced counts
ax=sb.distplot(adata8.obs['n_spliced'], kde=False)
plt.show()

ax=sb.distplot(adata8.obs['n_spliced'][adata8.obs['n_spliced']<2500], kde=False, bins=60)
plt.show()

ax=sb.distplot(adata8.obs['n_spliced'][adata8.obs['n_spliced']>10000], kde=False, bins=60)
plt.show()

Let's look at unspliced counts.

In [ ]:
#Thresholding decision: unspliced counts
ax=sb.distplot(adata8.obs['n_unspliced'], kde=False)
plt.show()

ax=sb.distplot(adata8.obs['n_unspliced'][adata8.obs['n_unspliced']<1500], kde=False, bins=60)
plt.show()

In [ ]:
#Thresholding decision: genes
ax=sb.distplot(adata8.obs['n_genes'], kde=False, bins=60)
plt.show()
ax=sb.distplot(adata8.obs['n_genes'][adata8.obs['n_genes']<1000], kde=False, bins=60)
plt.show()

In [ ]:
print("Before filtering cells:\n", adata8, "\n")

sc.pp.filter_cells(adata8, min_counts=1600)
sc.pp.filter_cells(adata8, max_counts=16000)
adata8 = adata8[adata8.obs['n_unspliced']>800]
sc.pp.filter_cells(adata8, min_genes=500)

print("After filtering cells:\n", adata8)

In [ ]:
sc.pl.violin(adata8, ['n_genes', 'n_counts', 'percent_mito'],
             jitter=0.4, multi_panel=True)

In [ ]:
sc.pl.scatter(adata8, 'n_genes', 'percent_mito')

In [ ]:
adata8 = adata8[adata8.obs['percent_mito'] < 0.3, :]
sc.pl.scatter(adata8, 'n_counts', 'n_genes', color='percent_mito', title='Percent mitochondrial genes')
print(adata8)

In [ ]:
sc.pl.scatter(adata8, 'n_genes', 'doublet_score', color='doublet', palette=['gray', 'orange'], legend_loc='none', title='Doublet')
adata8 = adata8[adata8.obs['doublet'] != '1']
adata8 = adata8[adata8.obs['doublet_score'] < 0.3]

In [ ]:
sc.pl.highest_expr_genes(adata8, n_top=20)
print(adata8)

## 3. Preprocessing

### 3.1 Normalisation

For the rest of the analysis we concatenate the samples into one object `adata`. First, we normalise the counts per cell (CPM normalisation) and log transform them. At this point we save the expression matrix to the `.raw` attribute of our object. It is important to keep the "raw" values for DE analysis or visulaising markers expression.   

There's the option to normalise all cells to 1e6 counts per cell (CPM normalisation). However, this would mask differences between cells of different sizes. Instead we normalise "to the median of total counts for observations (cells) before normalization". Layers (spliced, unspliced) are also normalised to the median of counts (`adata.X`).

In [ ]:
adata = adata0.concatenate(adata1, adata7, adata8,
                           batch_key='Sample', 
                           batch_categories=['healthy0', 'healthy1', 'healthy7', 'healthy8'])
# Apply workaround necessary due to bugs in code
adata.layers['spliced'] = adata.layers['spliced'].astype(float)
adata.layers['unspliced'] = adata.layers['unspliced'].astype(float)

scv.pp.filter_genes(adata, min_cells=5)
scv.pp.filter_genes(adata, min_counts=20)

scv.pp.normalize_per_cell(adata, max_proportion_per_cell=0.05)
scv.pp.log1p(adata)
adata.raw = adata

adata

In [ ]:
import loompy as lp

adata = adata0.concatenate(adata1, adata7, adata8,
                           batch_key='Sample', 
                           batch_categories=['healthy0', 'healthy1', 'healthy7', 'healthy8'])
# Apply workaround necessary due to bugs in code
adata.layers['spliced'] = adata.layers['spliced'].astype(float)
adata.layers['unspliced'] = adata.layers['unspliced'].astype(float)

scv.pp.filter_genes(adata, min_cells=5)
scv.pp.filter_genes(adata, min_counts=20)

# # path to loom file with basic filtering applied (this will be created in the "initial filtering" step below). Optional.
f_loom_path_scenic = "adata_raw.loom"

# create basic row and column attributes for the loom file:
row_attrs = {
    "Gene": np.array(adata.var_names) ,
}
col_attrs = {
    "CellID": np.array(adata.obs_names) ,
    "nGene": np.array( np.sum(adata.X.transpose()>0 , axis=0)).flatten() ,
    "nUMI": np.array( np.sum(adata.X.transpose() , axis=0)).flatten() ,
}
lp.create( f_loom_path_scenic, adata.X.transpose(), row_attrs, col_attrs)

### 3.2 Cell cycle

Before we can move on with regression, we need to assign a cell cycle phase to all the cells. Not only it is interesting to visulaise it later on, but we will use it to diminish differences between cycling cells. 

Specifically, we want to regress out the difference between *S phase* and *G2M phase* in order to remove cycling patterns which negatively influences the RNA velocity analysis. However, we don't want to remove all signals coming from a cell cycle phase since we are still interested in looking at progenitors of different cell lineages. 

We determine the cell cycle using a simple function checking for gene enrichment. We supply two lists of genes associated with a given phase of cell cycle (taken from the **literature**). The score is the average expression of a set of genes subtracted with the average expression of a reference set of genes. The reference set is randomly sampled from all the genes. 

`CC_difference` score is calculated as the difference between cycling and not cycling cells.  

In [ ]:
ProlifMarkers = [i for i in ProlifMarkers if i in adata.var_names]

sc.tl.score_genes(adata, ProlifMarkers, score_name='Proliferation')

In [ ]:
sc.tl.score_genes_cell_cycle(adata, s_genes=s_genes, g2m_genes=g2m_genes)
adata.obs['CC_difference'] = adata.obs['S_score'] - adata.obs['G2M_score']

### 3.3 Regression and highly variable genes (HVGs)

As we could have seen previously there are intrinsic differences between the samples (like the sequencing depth). Since we want those samples to represent the same system in the same condition (healthy colon), we can assume that most of the variance betweeen samples is technical. We get rid of that variation by regressing **number of counts** and **CC difference**. This is done using **linear regression**.

Finally, we can select Highly Variable Genes (HVGs). The remaining analysis will be done only on them. We keep all the genes (uncorrected) in the `.raw` attribute to use them for plotting and differential expression.

In [ ]:
sc.pp.highly_variable_genes(adata, n_top_genes=2000, subset=False)

In [ ]:
corrected = sc.api.pp.mnn_correct(adata[adata.obs['Sample']=='healthy0'], adata[adata.obs['Sample']=='healthy1'], 
                                  adata[adata.obs['Sample']=='healthy7'], adata[adata.obs['Sample']=='healthy8'],
                                  batch_key='Sample', batch_categories=['healthy0', 'healthy1', 'healthy7', 'healthy8'],
                                  var_subset=list(adata.var_names[adata.var['highly_variable']]), k=15, var_adj=True, 
                                  do_concatenate=True, save_raw=True, n_jobs=12)

In [ ]:
adata = corrected[0].copy()

We also subset our adata object to include only progenitor cells. We define those as cells being in cell cycle, so having **positive S and/or G2M scores**. We will use the progenitor dataset at a later part of the analysis.

In [ ]:
G2M_cells = [name for name in adata.obs_names if adata.obs['G2M_score'][name] > 0]
S_cells = [name for name in adata.obs_names if adata.obs['S_score'][name] > 0]
CC_cells = set(G2M_cells + S_cells)
adata_progen = adata[list(CC_cells),:].copy()

### 3.4 Dimentionality reduction and Principal Component analysis

Next, we can perform PCA...

In [ ]:
sc.pp.regress_out(adata, ['S_score', 'G2M_score'], n_jobs=12)

In [ ]:
sc.pp.highly_variable_genes(adata, n_top_genes=2000, subset=False)

In [ ]:
sc.tl.pca(adata, svd_solver='arpack')

...and plot it to look where is the most variance in the dataset coming from.

In [ ]:
# Change plotting settings for more visually pleasing.
scv.settings.set_figure_params('scvelo', dpi=150, vector_friendly=False)

features = ['Sample', 'log_counts', 'phase', 'percent_mito', 'percent_ribo', 'doublet_score']
sc.pl.pca(adata, color=features, ncols=2)

In [ ]:
sc.set_figure_params(fontsize=12, vector_friendly=True, dpi=150, dpi_save=300)

fig, (ax1, ax2, ax3, ax4) = plt.subplots(1, 4, figsize=(20,10))

sc.pl.pca(adata, color='Sample', cmap='YlOrRd', alpha=0.5, ax=ax1, show=False, size=15)
adata.uns['pca']['variance_ratio']
ax1.set_aspect(aspect=1)
ax1.set_xlabel('PC1 {}%'.format(round(adata.uns['pca']['variance_ratio'][0]*100, 2)))
ax1.set_ylabel('PC2 {}%'.format(round(adata.uns['pca']['variance_ratio'][1]*100, 2)))
fig = plt.gcf()
cbar_ax = fig.axes[-1]
cbar_ax.set_aspect(aspect=10)

sc.pl.pca(adata, color='LGR5', cmap='YlOrRd', ax=ax2, show=False, size=15)
ax2.set_aspect(aspect=1)
ax2.set_xlabel('PC1 {}%'.format(round(adata.uns['pca']['variance_ratio'][0]*100, 2)))
ax2.set_ylabel('PC2 {}%'.format(round(adata.uns['pca']['variance_ratio'][1]*100, 2)))
fig = plt.gcf()
cbar_ax = fig.axes[-1]
cbar_ax.set_aspect(aspect=10)

sc.pl.pca(adata, color='MUC2', cmap='YlOrRd', ax=ax3, show=False, size=15)
ax3.set_aspect(aspect=1)
ax3.set_xlabel('PC1 {}%'.format(round(adata.uns['pca']['variance_ratio'][0]*100, 2)))
ax3.set_ylabel('PC2 {}%'.format(round(adata.uns['pca']['variance_ratio'][1]*100, 2)))
fig = plt.gcf()
cbar_ax = fig.axes[-1]
cbar_ax.set_aspect(aspect=10)

sc.pl.pca(adata, color='CA1', cmap='YlOrRd', ax=ax4, show=False, size=15)
ax4.set_aspect(aspect=1)
ax4.set_xlabel('PC1 {}%'.format(round(adata.uns['pca']['variance_ratio'][0]*100, 2)))
ax4.set_ylabel('PC2 {}%'.format(round(adata.uns['pca']['variance_ratio'][1]*100, 2)))
fig = plt.gcf()
cbar_ax = fig.axes[-1]
cbar_ax.set_aspect(aspect=10)

fig.tight_layout()

fig.savefig('figures/PCA_supp-fig.pdf', bbox_inches='tight')

In [ ]:
features = ['LGR5', 'CEACAM1', 'MUC2', 'CA1', 'BEST4', 'KRT20', 'SCGN', 'LRMP']
sc.pl.pca_overview(adata, color=features, projection='3d', ncols=2)

We can see above that majority of the variance in the dataset is biological and comes from differences between cell lineages:
- Crypt top vs bottom (PC1)
- Goblet lineage vs colonocyte lineage (PC2)
- BEST4 colonocytes vs colonocyte lineage (PC3)

## 4. Graph embedding and clustering

### 4.1 Batch correction with bbknn

Regressing number of counts corrected the samples enough for them to align at the first few PCs. However, analysis like clustering are done in multi dimensional space, so we need to ensure the samples match together. This is done at the step of finding neighbors for each of the cells.

We calculate the neighbours using batch balanced k nearest neighbours. "Batch balanced kNN alters the kNN procedure to identify each cell’s top neighbours in each batch separately instead of the entire cell pool with no accounting for batch."

It is a very robust batch correction method than have been shown to work even better than mutual nearest neighbors. It is also much (much) faster than MNN. 

In [ ]:
sc.external.pp.bbknn(adata, batch_key='Sample', neighbors_within_batch=5, n_pcs=30,
                     approx=False, use_faiss=False, metric='euclidean')

### 4.2 Graph embedding

Afterwards, the kNN graph is used to create a joint embedding for the cells with `umap` (better than t-SNE).

In [ ]:
sc.tl.umap(adata)

We can now explore the embedding based on some known **gene markers** in the colon.

In [ ]:
features = ['Sample', 'log_counts', 'phase', 'percent_mito',
            'MUC2', 'CA1', 'CA2', 'LGR5', 'KRT20', 'LRMP', 'SCGN', 
            'CEACAM1', 'TFF1' , 'PCNA', 'MKI67', 'TFF3', 'BEST4', 'MALAT1']

sc.pl.umap(adata, color=features, use_raw=True, ncols=2)

In [ ]:
sc.tl.rank_genes_groups(adata, groupby='Celltype', method='wilcoxon', groups=['TA Goblet'], reference='Goblet')
sc.tl.filter_rank_genes_groups(adata, min_in_group_fraction=0.25, 
                               max_out_group_fraction=0.75, min_fold_change=1.1)
sc.pl.rank_genes_groups_dotplot(adata, n_genes=10, key='rank_genes_groups_filtered', 
                                standard_scale='var', color_map='bwr', dendrogram=False)

In [ ]:
gene='FCGBP'

sc.pl.umap(adata, color=gene, use_raw=True, cmap='YlOrRd')

In [ ]:
sc.set_figure_params(fontsize=12, vector_friendly=True, dpi=150, dpi_save=300)



In [ ]:
sc.pl.umap(adata, color='Sample', legend_loc='right margin', alpha=0.7, size=7, save='HealthySample', title='')

In [ ]:
features = ['LGR5', 'ASCL2', 'SOX4', 'HES6', 'ADH1C', 'GGH', 'CA1', 'SLC26A3', 'PLAC8', 'MS4A12', 'LYZ', 'LEFTY1', 'CA4', 'BEST4',
           'OTOP2', 'GUCA2B', 'RETNLB', 'TFF3', 'MUC2', 'REG4', 'TFF1', 'FCGBP', 'LRMP', 'FYB1', 'SCGN', 'PYY']
sc.set_figure_params(fontsize=12, vector_friendly=True, dpi=150, dpi_save=300)
sc.pl.umap(adata, color=features, use_raw=True, ncols=2, save='CloudHierarchyMarkers', cmap='YlOrRd')

In [ ]:
features = ['LGR5', 'CA1', 'SCGN', 'BEST4', 'LRMP', 'MUC2']
sc.set_figure_params(fontsize=12, vector_friendly=True, dpi=150, dpi_save=300)
sc.pl.umap(adata, color=features, use_raw=True, ncols=2, save='HealthyyMarkers', cmap='YlOrRd')

Using a list of genes taken from the literature we can also determine which cells are proliferating. 

In [ ]:
sc.pl.umap(adata, color=['ATOH1', 'SOX4'], cmap='YlOrRd', save='Healthy_SOX4ATOH')

In [ ]:
sc.pl.umap(adata, color=['SOX4', 'CLCA1'])

In [ ]:
sc.pl.umap(adata, color=['REG1A', 'TFF1'], cmap='Reds')

Markers for Crypt top populations

In [ ]:
sc.set_figure_params(fontsize=12, vector_friendly=True, dpi=150, dpi_save=300)

features=['CEACAM1', 'CEACAM5', 'SELENOP', 'PLAC8']
sc.pl.umap(adata, color=features, cmap='YlOrRd', save='Healthy_CryptTop', use_raw=True, ncols=2)

In [ ]:
sc.set_figure_params(fontsize=12, vector_friendly=True, dpi=150, dpi_save=300)
sc.pl.umap(adata, color='G2M_score', vmin=-0.45, vmax=0.45, cmap='bwr', title='G2M score', save='HealthyG2M')

In [ ]:
sc.set_figure_params(fontsize=12, vector_friendly=True, dpi=150, dpi_save=300)
sc.pl.umap(adata, color='S_score', vmin=-0.45, vmax=0.45, cmap='bwr', title='S score', save='HealthyS')

In [ ]:
sc.set_figure_params(fontsize=12, vector_friendly=True, dpi=150, dpi_save=300, figsize=[4,4])
sc.pl.umap(adata, color=['SOX4','LRMP'], cmap='YlOrRd', ncols=2, save='Healthy-tuftorigin')

In [ ]:
sc.set_figure_params(fontsize=12, vector_friendly=True, dpi=150, dpi_save=300, figsize=[4,4])
sc.pl.umap(adata, color=['MUC2','TFF3', 'PCNA'], cmap='YlOrRd', save='HEALTHY-secretprogen', ncols=1)

### 4.3 Finding clusters and celltypes 

We look for clusters in the cell graph using unsupervised clustering method `leiden` (better than louvain).

In [ ]:
sc.tl.leiden(adata, resolution=1.3)

In [ ]:
sc.tl.leiden(adata, resolution=0.2, restrict_to=('leiden', ['9']))

In [ ]:
adata.obs['leiden'] = adata.obs['leiden'].replace("1", "0")
adata.obs['leiden'] = adata.obs['leiden'].replace("2", "1")
adata.obs['leiden'] = adata.obs['leiden'].replace("5", "1")
adata.obs['leiden'] = adata.obs['leiden'].replace("3", "2")
adata.obs['leiden'] = adata.obs['leiden'].replace("4", "3")
adata.obs['leiden'] = adata.obs['leiden'].replace("6", "4")
adata.obs['leiden'] = adata.obs['leiden'].replace("7", "5")
adata.obs['leiden'] = adata.obs['leiden'].replace("8", "6")
adata.obs['leiden'] = adata.obs['leiden'].replace("9", "7")
adata.obs['leiden'] = adata.obs['leiden'].replace("10,0", "8")
adata.obs['leiden'] = adata.obs['leiden'].replace("10,1", "9")
adata.obs['leiden'] = adata.obs['leiden'].replace("11", "10")
adata.obs['leiden'] = adata.obs['leiden'].replace("12", "11")
adata.obs['leiden'] = adata.obs['leiden'].replace("13", "12")

In [ ]:
sc.set_figure_params(fontsize=12, vector_friendly=True, dpi=150, dpi_save=300)
sc.pl.umap(adata, color='leiden_R', legend_loc='on data', save='Healthy_leiden', title='')

In [ ]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
sc.tl.rank_genes_groups(adata, groupby='leiden', method='wilcoxon')
sc.tl.filter_rank_genes_groups(adata, min_in_group_fraction=0.25, 
                               max_out_group_fraction=0.75, min_fold_change=1.5)
sc.pl.rank_genes_groups_dotplot(adata, n_genes=10, key='rank_genes_groups_filtered', 
                                standard_scale='var', color_map='bwr', dendrogram=False)

Following expression of the marker genes we can reassign and rename clusters to more meaningful structures that should correspond to celltypes. 

In [ ]:
adata.obs['Celltype'] = adata.obs['leiden_R']
adata.obs['Celltype'] = adata.obs['Celltype'].replace("0", "TA Colonocytes")
adata.obs['Celltype'] = adata.obs['Celltype'].replace("1", "TA Goblet")
adata.obs['Celltype'] = adata.obs['Celltype'].replace("2", "TA Colonocytes")
adata.obs['Celltype'] = adata.obs['Celltype'].replace("3", "TA Goblet")
adata.obs['Celltype'] = adata.obs['Celltype'].replace("4", "BEST4+")
adata.obs['Celltype'] = adata.obs['Celltype'].replace("5", "TA Colonocytes")
adata.obs['Celltype'] = adata.obs['Celltype'].replace("6", "TA BEST4+")
adata.obs['Celltype'] = adata.obs['Celltype'].replace("7", "Goblet")
adata.obs['Celltype'] = adata.obs['Celltype'].replace("8", "Colonocytes")
adata.obs['Celltype'] = adata.obs['Celltype'].replace("9,0", "Stem")
adata.obs['Celltype'] = adata.obs['Celltype'].replace("9,1", "TA SOX4+")
adata.obs['Celltype'] = adata.obs['Celltype'].replace("10", "TA Colonocytes")
adata.obs['Celltype'] = adata.obs['Celltype'].replace("11", "CT BEST4+")
adata.obs['Celltype'] = adata.obs['Celltype'].replace("12", "CT Colonocytes")
adata.obs['Celltype'] = adata.obs['Celltype'].replace("13", "Tuft")
adata.obs['Celltype'] = adata.obs['Celltype'].replace("14", "CT Goblet")
adata.obs['Celltype'] = adata.obs['Celltype'].replace("15", "EECs")

adata.obs['Celltype'] = adata.obs['Celltype'].astype('category')
Celltype_order = ['Stem',
                  'TA SOX4+',
                  'TA Colonocytes', 'Colonocytes', 'CT Colonocytes',
                  'TA BEST4+', 'BEST4+', 'CT BEST4+', 
                  'TA Goblet', 'Goblet', 'CT Goblet',                  
                  'EECs' ,'Tuft']
adata.obs['Celltype'].cat.reorder_categories(Celltype_order, inplace=True)

Assign colors to celltypes

In [ ]:
vega_colors = np.array(sc.pl.palettes.vega_20_scanpy)

celltype_colors = np.zeros(len(set(adata.obs['Celltype'])))
celltype_colors = celltype_colors.astype('U7')

celltype_colors[[0]] =  vega_colors[[1]] # Stem color / orange
celltype_colors[[1]] = '#FFCC00' # TA Goblet SOX4+ colors / yellow
celltype_colors[[2, 3, 4]] = vega_colors[[12, 10, 15]]  # Colono colors / reds
celltype_colors[[5, 6, 7]] = vega_colors[[0, 8, 17]]  # BEST4 colors / blues
celltype_colors[[8, 9, 10]] = vega_colors[[2, 7, 11]]  # Goblet colors / greens
celltype_colors[[11]] = '#E53333'  # EECs / red
celltype_colors[[12]] = '#A9A9A9'  # Tuft / grey

adata.uns['Celltype_colors'] = celltype_colors

In [ ]:
vega_colors[[2]]

Plot celltypes with new coloring

In [ ]:
adata.obs_names = pd.Index([i[:-9] for i in list(adata.obs_names.values)])

In [ ]:
sc.set_figure_params(fontsize=12, vector_friendly=True, dpi=150, dpi_save=300)
sc.pl.umap(adata, color='Celltype', legend_loc='right margin', alpha=0.7, size=10, save='HealthyCelltype', title='')

Differential expression analysis between celltypes

In [ ]:
adata.write("healthy_postSCENIC.h5ad")

In [ ]:
Celltype_order = ['Stem',
                  'TA SOX4+',
                  'TA Colonocytes', 'Colonocytes', 'CT Colonocytes',
                  'TA BEST4+', 'BEST4+', 'CT BEST4+', 
                  'TA Goblet', 'Goblet', 'CT Goblet',                  
                  'EECs' ,'Tuft']

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
sc.tl.rank_genes_groups(adata, groupby='Celltype', method='wilcoxon')
sc.tl.filter_rank_genes_groups(adata, min_in_group_fraction=0.5, 
                               max_out_group_fraction=0.5, min_fold_change=1.5)
RevCelltype_order = Celltype_order
RevCelltype_order.reverse()
#adata.obs['Celltype'].cat.reorder_categories(RevCelltype_order, inplace=True)
sc.pl.rank_genes_groups_dotplot(adata, n_genes=20, key='rank_genes_groups_filtered', groupby='Celltype',
                                standard_scale='var', color_map='bwr', dendrogram=False, save='DE_healthy',
                                figsize=(40,3))

In [ ]:
adata.obs['Celltype'].cat.reorder_categories(Celltype_order, inplace=True)

# Expression summary 

In [ ]:
t = pd.DataFrame(data=adata.raw.X.toarray(), index=adata.obs_names, columns=adata.raw.var_names)
Expression_SOX4 = {'Celltype':adata.obs['Celltype'], 'Expression':t['SOX4']}
Expression_ATOH1 = {'Celltype':adata.obs['Celltype'], 'Expression':t['ATOH1']}


Expression_SOX4 = pd.DataFrame(data=Expression_SOX4)
Expression_ATOH1 = pd.DataFrame(data=Expression_ATOH1)

In [ ]:
Expression_SOX4.to_csv('Expression_SOX4.csv')
Expression_ATOH1.to_csv('Expression_ATOH1.csv')

## 5. Trajectory analysis

### 5.1 PAGA

Redo clustering with higher resolution.

In [ ]:
adata.obs['SubCelltype'] = adata.obs['Celltype']
for celltype in adata.obs['Celltype'].cat.categories.tolist():
    sc.tl.leiden(adata, resolution=0.69, restrict_to=('SubCelltype', [celltype]), key_added='SubCelltype')

In [ ]:
sc.set_figure_params(fontsize=12, vector_friendly=True, dpi=150, dpi_save=300)
sc.pl.umap(adata, color='SubCelltype', legend_loc='none', save='Healthy_subcelltype', title='', legend_fontweight='light')

Run PAGA on smaller (higher resolution) clusters, while keeping the information about the celltypes (coloring).

In [ ]:
del adata.obsm['X_diffmap']

In [ ]:
sc.tl.paga(adata, groups='SubCelltype')

In [ ]:
sc.pl.paga(adata, color='Celltype', threshold=0.7, 
           node_size_scale=0.6, edge_width_scale=0.1, node_size_power=0.2, 
           layout='fr', text_kwds={'alpha':0}, frameon=False)#, save='Healthy')

In [ ]:
sc.pl.paga(adata, color='TFF3', threshold=0.7, 
           node_size_scale=0.6, edge_width_scale=0.1, node_size_power=0.2, 
           layout='fr', text_kwds={'alpha':0})

In [ ]:
sc.pl.paga(adata, color='phase', threshold=0.4, 
           node_size_scale=0.6, edge_width_scale=0.2, node_size_power=0.2, 
           layout='fr', text_kwds={'alpha':0})

In [ ]:
sc.pl.paga(adata, color=['RARRES2', 'CA7', 'SOX4', 'HES4'], cmap='Reds', threshold=0.75, 
           node_size_scale=0.6, edge_width_scale=0.1, node_size_power=0.2, 
           layout='fr', text_kwds={'alpha':0})

In [ ]:
sc.pl.paga(adata, color=['TFF3', 'BEST4'], cmap='Reds', threshold=0.4, 
           node_size_scale=0.6, edge_width_scale=0.2, node_size_power=0.2, 
           layout='fr', text_kwds={'alpha':0})

In [ ]:
sc.pl.paga(adata, color=['SOX4', 'ATOH1'], cmap='Reds', threshold=0.75, 
           node_size_scale=0.6, edge_width_scale=0.1, node_size_power=0.2, 
           layout='fr', text_kwds={'alpha':0})

### 5.2 Graph embedding based on PAGA

In [ ]:
sc.set_figure_params(fontsize=12, vector_friendly=True, dpi=150, dpi_save=300)

In [ ]:
sc.tl.draw_graph(adata, init_pos='paga')

In [ ]:
sc.pl.draw_graph(adata, color='Celltype', title='',
                 size=10, alpha=0.7, legend_fontsize=6, frameon=False, edges=True, save='Healthy_edges', legend_loc='none')

In [ ]:
sc.pl.draw_graph(adata, color='doublet_score', 
                 size=10, legend_fontsize=6, frameon=False, edges=True, save='test')

In [ ]:
sc.pl.draw_graph(adata, color='TFF3', 
                 size=30, legend_fontsize=6, frameon=False, edges=False)

In [ ]:
sc.pl.draw_graph(adata, color='SOX4',
                 size=30, legend_fontsize=6, frameon=False, edges=False, save='HEALTHYSOX4')

In [ ]:
sc.pl.draw_graph(adata, color='ATOH1',
                 size=30, legend_fontsize=6, frameon=False, edges=False)

In [ ]:
sc.pl.draw_graph(adata, color=['TFF3', 'CA7'], size=50, alpha=0.75, legend_fontsize=6, 
                 frameon=False, edges=False, save='TFF3vsCA7')

In [ ]:
sc.pl.draw_graph(adata, color=['LRMP', 'CA2'], size=50, alpha=0.75, legend_fontsize=6, 
                 frameon=False, edges=False)

### 5.3 Diffusion-based pseudotime

In [ ]:
# Set one of the stem cells as the starting point for the pseudotime.
# Biased - compare with latent time. 
adata.uns['iroot'] = np.flatnonzero(adata.obs['Celltype']  == 'Stem')[0]

sc.tl.diffmap(adata)
sc.tl.dpt(adata)

In [ ]:
scv.pl.scatter(adata, color='dpt_pseudotime', legend_loc='on data', 
                 size=100, fontsize=24, frameon=False, 
                 cmap='viridis', save='HEALTHYpseudo', title='Pseudotime')

## X. Other tools

### X.1 Phenograph 

In [ ]:
result = sc.external.tl.phenograph(adata.obsm['X_pca'], k = 35, primary_metric='correlation', n_jobs=12, directed=True)

In [ ]:
adata.obs['pheno'] = pd.Categorical(result[0])

In [ ]:
sc.pl.umap(adata, color='pheno', legend_loc='on data', alpha=0.7, size=10, title='')

### X.2 Embedding density 

In [ ]:
sc.pl.umap(adata, color='Sample', size=30, alpha=0.25)

In [ ]:
sc.tl.embedding_density(adata, basis='umap', groupby='Sample')

In [ ]:
sc.pl.embedding_density(adata, basis='umap', key='umap_density_Sample', group='healthy8')

In [ ]:
sc.tl.embedding_density(adata, basis='umap', groupby='phase')

In [ ]:
sc.pl.embedding_density(adata, basis='umap', key='umap_density_phase', group='G2M')

### X.3 MAGIC

In [ ]:
adata_magic = sc.external.pp.magic(adata, name_list='all_genes', knn=5, n_jobs=12, copy=True)

In [ ]:
adata_magic

In [ ]:
sc.pl.scatter(adata_magic, basis='magic', color='Celltype')

## XX.1 Fraction of cells statistics

In [ ]:
celltype_sizes = adata.obs['Celltype'].value_counts()

sample_summary = pd.DataFrame(adata.obs.groupby(['Celltype'])['Sample'].value_counts()).rename(columns={'Sample': "Counts"})

In [ ]:
sample_summary = sample_summary.reset_index()

sample_summary['Fraction'] = [sample_summary.loc[index, 'Counts']/celltype_sizes[sample_summary.loc[index, 'Celltype']] 
                                  for index in sample_summary.index]
del sample_summary['Counts']

sample_summary.to_csv('healthy_sample_summary.csv')

## 6. Velocity

### 6.1 Velocity computation 

Preprocessing. Compute first order moments for spliced and unspliced counts. Recover the full splicing kinetics of specified genes. The model infers transcription rates, splicing rates, degradation rates, as well as cell-specific latent time and transcriptional states. The most time consuming step of the pipeline. Consider stochastic model for significant speed up. 

In [ ]:
scv.pp.moments(adata, n_pcs=30, n_neighbors=20)
scv.tl.recover_dynamics(adata)

In [ ]:
scv.tl.velocity(adata, mode='dynamical', groupby='Celltype')
scv.tl.velocity_graph(adata)#, n_neighbors=20, approx=True)#, gene_subset=list(adata.var_names[adata.var['highly_variable']]))
scv.tl.recover_latent_time(adata)

In [ ]:
scv.pl.scatter(adata, x='latent_time', y=['FCGBP', 'CLCA1', 'TFF3'], fontsize=16, size=100,
               n_convolve=None, frameon=False, legend_loc='none', color='Celltype')

In [ ]:
sc.pl.umap(adata, color=['root_cells', 'end_points'], cmap='Reds', ncols=1,
               legend_fontsize=50, frameon=False, size=50)

In [ ]:
scv.pl.velocity_graph(adata, color='Celltype', legend_loc='right margin', perc=90,
                     basis='draw_graph_fa', n_neighbors=None, title='Transition graph based on velocity')

Velocity calculation based on the dynamical model. It adapts RNA velocity to widely varying specifications such as non-stationary populations, as it does not rely on the restrictions of a common splicing rate or steady states to be sampled. We also use latent time as a regularization for velocity.

### 6.3 Plotting velocity on embeddings

In [ ]:
scv.pl.velocity_embedding_grid(adata, basis='umap', 
                                 legend_fontsize=10, title='', 
                                 smooth=.52, min_mass=0, color='Celltype',
                                 alpha=0.7, size=30, fontsize=30, legend_loc='none', save="Healthy_grid")

In [ ]:
scv.set_figure_params(fontsize=12, vector_friendly=True, dpi=150, dpi_save=300, figsize=[4,4], format='png')

scv.pl.velocity_embedding_stream(adata, basis='umap', 
                                 legend_fontsize=10, title='', 
                                 smooth=.52, min_mass=0, color='Celltype',
                                 legend_loc='none', save='healthy_stream')

In [ ]:
gene = "ADH1C"
scv.pl.velocity_embedding_stream(adata, basis='umap', cmap='YlOrRd',
                                 legend_fontsize=10, title='{} expression'.format(gene), 
                                 smooth=.52, min_mass=0, color=gene,
                                 alpha=0.7, size=30, fontsize=30, legend_loc='none', save="{}velocityHEALTHY".format(gene))

In [ ]:
sc.pl.pca_loadings(adata, components=[1,2,3,4,5,6,7, 8, 9, 10, 11, 12, 13, 14])

In [ ]:
scv.pl.velocity_embedding_grid(adata, color='Celltype', legend_loc='none', basis='pca', components=[2,5],
                          alpha=0.7, size=60, title='', legend_fontsize=20, cmap='YlOrRd')

In [ ]:
scv.pl.velocity_embedding(adata, color='Celltype', legend_loc='none', basis='umap',
                          alpha=0.7, size=60, title='', arrow_size=5, arrow_length=5, 
                          save='HealthyArrows', legend_fontsize=20)

In [ ]:
scv.pl.velocity_embedding(adata, color='Celltype', legend_loc='none', basis='draw_graph_fa',
                          alpha=0.7, size=60, title='', arrow_size=5, arrow_length=5, 
                          save='HealthyArrowsFA', legend_fontsize=20)

In [ ]:
scv.pl.velocity_embedding(adata, color=['LRMP','SOX4'], legend_loc='right margin', basis='draw_graph_fa',
                          alpha=0.7, size=60, title='', arrow_size=5, arrow_length=5, legend_fontsize=20, save='test')

In [ ]:
scv.pl.velocity_embedding(adata, color=['TFF3', 'ASCL2'], legend_loc='right margin', basis='draw_graph_fa',
                          alpha=0.7, size=60, title='', arrow_size=5, arrow_length=5, legend_fontsize=20)

In [ ]:
scv.pl.velocity_embedding(adata, color='phase', legend_loc='right margin', basis='draw_graph_fa', color_map='bwr',
                          alpha=0.7, size=60, arrow_size=5, arrow_length=5, legend_fontsize=20, save='test1')

In [ ]:
scv.pl.velocity_embedding(adata[adata.obs['Celltype']!='TA Goblet'], color='SOX4', legend_loc='none', basis='umap',
                          alpha=0.7, size=60, title='', arrow_size=5, arrow_length=5, legend_fontsize=20, save='test2')

In [ ]:
sc.pl.umap(adata, color='LGR5', size=30)

In [ ]:
scv.pl.velocity_embedding(adata, basis='draw_graph_fa', color='Celltype', legend_loc='none', 
                          alpha=0.7, size=60, title='', arrow_size=5, arrow_length=5, legend_fontsize=20, 
                          save='HealthyArrows_fa')

In [ ]:
scv.pl.velocity_embedding(adata, basis='draw_graph_fa', color='phase', legend_loc='none', 
                          alpha=0.7, size=60, title='', arrow_size=5, arrow_length=5, legend_fontsize=20)

In [ ]:
scv.tl.velocity_confidence(adata)

In [ ]:
scv.pl.scatter(adata, basis='umap', color='velocity_confidence', fontsize=24, size=100, colorbar=True, rescale_color=[0,1], color_map='RdBu_r')
scv.pl.scatter(adata, basis='umap', color='velocity_confidence_transition', fontsize=24, size=100, colorbar=True, rescale_color=[0,1], color_map='RdBu_r')

### 6.5 Latent time

The latent time is more accurate than the diffusion base pseudotime and does not require us to specify the root cell manually. In the plot below color is scaled between 2nd and 98th percentile to normalise extreme values in continous coloring. 

In [ ]:
scv.set_figure_params(fontsize=12, vector_friendly=True, dpi=150, dpi_save=300, figsize=[4,4])

scv.pl.scatter(adata, color='latent_time', fontsize=24, size=30, basis='umap',
               color_map='gnuplot', perc=[2, 98], colorbar=True, rescale_color=[0,1],
               title='', save='HealthyTime')

### 6.6 PAGA with velocity

In [ ]:
sc.tl.paga(adata, use_rna_velocity=False, groups='Celltype')
sc.tl.paga(adata, use_rna_velocity=True, groups='Celltype')

In [ ]:
sc.pl.paga(adata, color='Celltype', threshold=0.0001, 
           node_size_scale=0.6, edge_width_scale=0.5, node_size_power=0.3, 
           layout='fr', #text_kwds={'alpha':0}, 
           transitions='transitions_confidence', arrowsize=20, dashed_edges = "connectivities", save='velocityTransitionCelltype')

## 7. Analysis of the progentior population 

We want to have a deeper look at the population of **progenitors**. Our main quesiton is whether there are proliferating cells who are already commited to a given cell lineage. Or do we see just a single, general progenitor population. For that reason we have subsetted our dataset to include only cycling cells (with either *S score* or *G2M score* positive). Those cells should represent the proliferating progenitor population that resides in the TA compartment.  

In [ ]:
adata_progen = sc.read('healthy_progen_velo.h5ad')

In [ ]:
adata_progen

This subset of the data is analysed in the same way as before, with one significant change. Now we are regressing the cell cycle effect fully (both *S score* and *G2M score*). This is done so that we don't look at similarities between cells just because they are in the same cell cycle. 

## 7.1 Preprocessing

In [ ]:
sc.pp.regress_out(adata_progen, ['S_score', 'G2M_score'], n_jobs=12)
sc.pp.highly_variable_genes(adata_progen, n_top_genes=2000, subset=False)

## 7.2 PCA

In [ ]:
sc.tl.pca(adata_progen, svd_solver='arpack')

In [ ]:
scv.set_figure_params(fontsize=12, vector_friendly=True, dpi=150, dpi_save=300, figsize=[4,4], format='png')
features = ['Sample', 'log_counts', 'percent_mito', 'phase']
sc.pl.pca(adata_progen, color=features, ncols=2)

In [ ]:
features = ['MUC2', 'CA2', 'LGR5', 'CEACAM1', 'BEST4', 'KRT20']
sc.pl.pca_overview(adata_progen, color=features, ncols=2)

## 7.3 UMAP

In [ ]:
sc.external.pp.bbknn(adata_progen, batch_key='Sample', neighbors_within_batch=5, n_pcs=30,
                     approx=False, use_faiss=False, metric='euclidean')

In [ ]:
sc.tl.umap(adata_progen)

In [ ]:
features = ['Sample', 'log_counts', 'percent_mito', 'phase', 'Proliferation']

sc.pl.umap(adata_progen, color=features, use_raw=True, ncols=2)

In [ ]:
features = ['MUC2', 'BEST4', 'CA2', 'SOX4', 'LGR5', 'KRT20', 'LRMP', 'SCGN', 'TFF3', 'CA7', 'ETS2', 'CDK6']

sc.pl.umap(adata_progen, color=features, use_raw=True, ncols=2)

In [ ]:
sc.pl.umap(adata_progen, color=['S_score','G2M_score'], cmap='bwr')

In [ ]:
sc.pl.umap(adata_progen, color=['LGR5','ASCL2', 'SOX4'], cmap='YlOrRd')

## 7.4 Clustering

In [ ]:
sc.tl.leiden(adata_progen, resolution=0.95)

In [ ]:
sc.tl.leiden(adata_progen, resolution=0.6, restrict_to=('leiden', ['1']))

In [ ]:
sc.tl.leiden(adata_progen, resolution=0.3, restrict_to=('leiden_R', ['2']))

In [ ]:
sc.tl.leiden(adata_progen, resolution=0.3, restrict_to=('leiden_R', ['7']))

In [ ]:
sc.pl.umap(adata_progen, color='leiden_R', legend_loc='on data')

In [ ]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
sc.tl.rank_genes_groups(adata_progen, groupby='leiden', method='wilcoxon')
sc.tl.filter_rank_genes_groups(adata_progen, min_in_group_fraction=0.25, 
                               max_out_group_fraction=0.75, min_fold_change=1.5)
sc.pl.rank_genes_groups_dotplot(adata_progen, n_genes=10, key='rank_genes_groups_filtered', 
                                standard_scale='var', color_map='bwr', dendrogram=False)

In [ ]:
adata_progen.obs['Celltype'] = adata_progen.obs['leiden_R']
adata_progen.obs['Celltype'] = adata_progen.obs['Celltype'].replace("0", "TA Colonocytes")
adata_progen.obs['Celltype'] = adata_progen.obs['Celltype'].replace("1,0", "TA Goblet")
adata_progen.obs['Celltype'] = adata_progen.obs['Celltype'].replace("1,1", "TA Goblet")
adata_progen.obs['Celltype'] = adata_progen.obs['Celltype'].replace("1,2", "TA Goblet")
adata_progen.obs['Celltype'] = adata_progen.obs['Celltype'].replace("1,3", "Goblet")
adata_progen.obs['Celltype'] = adata_progen.obs['Celltype'].replace("2,0", "TA BEST4+")
adata_progen.obs['Celltype'] = adata_progen.obs['Celltype'].replace("2,1", "BEST4+")
adata_progen.obs['Celltype'] = adata_progen.obs['Celltype'].replace("3", "Stem")
adata_progen.obs['Celltype'] = adata_progen.obs['Celltype'].replace("4", "TA Colonocytes")
adata_progen.obs['Celltype'] = adata_progen.obs['Celltype'].replace("5", "TA Colonocytes")
adata_progen.obs['Celltype'] = adata_progen.obs['Celltype'].replace("6", "Colonocytes")
adata_progen.obs['Celltype'] = adata_progen.obs['Celltype'].replace("7,0", "TA Goblet")
adata_progen.obs['Celltype'] = adata_progen.obs['Celltype'].replace("7,1", "TA SOX4+")
adata_progen.obs['Celltype'] = adata_progen.obs['Celltype'].replace("8", "TA Tuft")
adata_progen.obs['Celltype'] = adata_progen.obs['Celltype'].replace("9", "TA EECs")


adata_progen.obs['Celltype'] = adata_progen.obs['Celltype'].astype('category')
Celltype_order = ['Stem',
                  'TA Colonocytes', 'Colonocytes',
                  'TA BEST4+', 'BEST4+',
                  'TA SOX4+',
                  'TA Goblet', 'Goblet',
                  'TA EECs', 'TA Tuft']
adata_progen.obs['Celltype'].cat.reorder_categories(Celltype_order, inplace=True)

In [ ]:
vega_colors = np.array(sc.pl.palettes.vega_20_scanpy)

celltype_colors = np.zeros(len(set(adata_progen.obs['Celltype'])))
celltype_colors = celltype_colors.astype('U7')

celltype_colors[[0]] =  vega_colors[[1]] # Stem color / orange
celltype_colors[[1, 2]] = vega_colors[[12, 10]]  # Colono colors / reds
celltype_colors[[3, 4]] = vega_colors[[0, 8]]  # BEST4 colors / blues
celltype_colors[[5]] = '#FFCC00' # TA SOX4+ colors / darkgreen
celltype_colors[[6, 7]] = vega_colors[[2, 7]]  # Goblet colors / greens
celltype_colors[[8]] = '#E53333'  # TA EECs / red
celltype_colors[[9]] = '#A9A9A9'  # TA Tuft / grey

adata_progen.uns['Celltype_colors'] = celltype_colors

In [ ]:
sc.set_figure_params(fontsize=12, vector_friendly=True, dpi=150, dpi_save=300)
sc.pl.umap(adata_progen, color='Celltype', legend_loc='right margin', save='ProgenCelltype')

## Cell cycle comparison

In [ ]:
#scores_s = {'Celltype':adata[adata.obs['S_score'] > 0].obs['Celltype'], 'Score':adata[adata.obs['S_score'] > 0].obs['S_score']}
#scores_g2m = {'Celltype':adata[adata.obs['G2M_score'] > 0].obs['Celltype'], 'Score':adata[adata.obs['G2M_score'] > 0].obs['G2M_score']}

#scores_prolif = {'Celltype':adata[adata.obs['Proliferation'] > 0].obs['Celltype'], 'Score':adata[adata.obs['Proliferation'] > 0].obs['Proliferation']}


scores_s = {'Celltype':adata.obs['Celltype'], 'Score':adata.obs['S_score']}
scores_g2m = {'Celltype':adata.obs['Celltype'], 'Score':adata.obs['G2M_score']}

scores_s = pd.DataFrame(data=scores_s)
scores_g2m = pd.DataFrame(data=scores_g2m)
#scores_prolif = pd.DataFrame(data=scores_prolif)

In [ ]:
scores_s.to_csv('scores_s.csv')
scores_g2m.to_csv('scores_g2m.csv')
#scores_prolif.to_csv('scores_prolif.csv')

In [ ]:
sc.pl.umap(adata, color=['G2M_score', 'S_score', 'Proliferation'], cmap='YlOrRd', ncols=1)

In [ ]:
adata.obs['Condition'] = 'HEALTHY'
sample_celltype_sizes = adata.obs.groupby(['Sample'])['Celltype'].value_counts()
phase_summary = pd.DataFrame(adata.obs.groupby(['Condition', 'Sample', 'Celltype'])['phase'].value_counts()).rename(columns={'phase': "Counts"})

In [ ]:
phase_summary = phase_summary.reset_index()

phase_summary['Percentage'] = [phase_summary.loc[index, 'Counts']*100/sample_celltype_sizes[phase_summary.loc[index, 'Sample'], phase_summary.loc[index, 'Celltype']] 
                                  for index in phase_summary.index]

phase_summary.to_csv('healthy_phase_summary.csv')

## 7.5 PAGA

In [ ]:
adata_progen.obs['SubCelltype'] = adata_progen.obs['Celltype']
for celltype in adata_progen.obs['Celltype'].cat.categories.tolist():
    sc.tl.leiden(adata_progen, resolution=0.6, restrict_to=('SubCelltype', [celltype]), key_added='SubCelltype')

In [ ]:
sc.pl.umap(adata_progen, color='SubCelltype', legend_loc='right margin')

In [ ]:
sc.tl.paga(adata_progen, groups='SubCelltype')

In [ ]:
sc.pl.paga(adata_progen, color='Celltype', threshold=0.75, fontsize=5, 
           node_size_scale=0.7, edge_width_scale=0.15, node_size_power=0.2, 
           layout='fr', text_kwds={'alpha':0}, frameon=False, save='HealthyPROGEN')

In [ ]:
sc.pl.paga(adata_progen, color='CA7', threshold=0.75, fontsize=5, 
           node_size_scale=0.7, edge_width_scale=0.15, node_size_power=0.2, 
           layout='fr', text_kwds={'alpha':0}, frameon=False)

In [ ]:
sc.pl.paga(adata_progen, color=['MUC2', 'CLCA1'], threshold=0.5, fontsize=5, 
           node_size_scale=0.7, edge_width_scale=0.15, node_size_power=0.2, 
           layout='fr', text_kwds={'alpha':0}, frameon=False)

In [ ]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
sc.tl.rank_genes_groups(adata_progen, groupby='Celltype', method='wilcoxon')
sc.tl.filter_rank_genes_groups(adata_progen, min_in_group_fraction=0.25, 
                               max_out_group_fraction=0.75, min_fold_change=1.5)
sc.pl.rank_genes_groups_dotplot(adata_progen, n_genes=10, key='rank_genes_groups_filtered', 
                                standard_scale='var', color_map='bwr', dendrogram=False)

## 7.6 FA graph embedding

In [ ]:
sc.tl.draw_graph(adata_progen, init_pos='paga')

In [ ]:
sc.pl.draw_graph(adata_progen, color='Celltype', size=50, 
                 legend_fontsize=12, frameon=False, edges=False, title="", save='PROGEN')

In [ ]:
sc.pl.draw_graph(adata_progen, color='KLK3', size=50,cmap='Reds',
                 legend_fontsize=12, frameon=False, edges=False)

In [ ]:
sc.pl.draw_graph(adata_progen, color=['MUC2', 'CLCA1', 'FCGBP', 'ITLN1', 'WFDC2', 'KLK1', 'SPINK4', 'SELENOM'],
                 size=50, alpha=0.8,
                 legend_fontsize=12, frameon=False, edges=False, ncols=2)

In [ ]:
sc.pl.draw_graph(adata_progen, color=['TFF3', 'CA7', 'MUC2', 'SOX4'], size=50, alpha=0.8,
                 legend_fontsize=12, frameon=False, edges=False, ncols=2, save='PROGENgobletTRANSmarkers')

## 7.7 Velocity

In [ ]:
scv.pp.moments(adata_progen, n_pcs=30, n_neighbors=20)
scv.tl.recover_dynamics(adata_progen)

In [ ]:
scv.tl.velocity(adata_progen, mode='dynamical', groupby='Celltype')
scv.tl.velocity_graph(adata_progen)#, n_neighbors=12)
scv.tl.recover_latent_time(adata_progen)

In [ ]:
scv.set_figure_params(fontsize=12, vector_friendly=True, dpi=150, dpi_save=300, figsize=[4,4], format='png')

scv.pl.velocity_embedding_stream(adata_progen, basis='umap', cmap='Reds',
                                 legend_fontsize=8, title='', 
                                 smooth=.9, min_mass=0.9, color='Celltype',
                                 alpha=0.9, size=50, fontsize=30, legend_loc='none', save='HealthyPROGENstream')

In [ ]:
sc.pl.violin(adata, keys='GNG4', groupby='Celltype', rotation=45, use_raw=True)

In [ ]:
scv.pl.velocity_embedding(adata_progen, basis='umap',
                                 legend_fontsize=8, 
                                 color='Celltype', arrow_size=5, arrow_length=5,
                                 alpha=0.7, size=60, fontsize=30, legend_loc='none')

In [ ]:
scv.pl.velocity_embedding(adata_progen, basis='draw_graph_fa', color='PKN1', legend_loc='none', 
                          alpha=0.7, size=60, arrow_size=5, arrow_length=5, legend_fontsize=20, 
                          save='PROGENArrows_fa')

In [ ]:
scv.pl.velocity_graph(adata_progen, color='Celltype', legend_loc='none', perc=90, 
                     basis='draw_graph_fa', n_neighbors=None, title='', save='PROGENtransitions')

In [ ]:
sc.pl.umap(adata_progen, color='SubCelltype', legend_loc='on data')

In [ ]:
scv.tl.rank_velocity_genes(adata_progen, match_with='Celltype')

scv.pl.velocity_embedding_stream(adata_progen, basis='umap', color="velocity_clusters", legend_loc='right margin',
                                 alpha=0.75, size=75)

pd.DataFrame(adata_progen.uns['rank_velocity_genes']['names']).head(n=10)

In [ ]:
scv.pl.velocity(adata_progen, var_names=['TLE4', 'GNG4', 'KPNA2', 'CTSZ', 'HERPUD1', 'C4orf48', 'TMC6', 'CCNJL', 'ANO7'], colorbar=True, ncols=1, layers=['velocity', 'Ms'], use_raw=False, cmap='RdBu_r', color_map='RdBu_r')



In [ ]:
top_genes = adata_progen.var_names[adata_progen.var.fit_likelihood.argsort()[::-1]][:300]
scv.pl.heatmap(adata_progen, var_names=top_genes, tkey='latent_time', n_convolve=100, col_color='Celltype', color_map='RdBu_r', xkey='Ms')

Stem velocity genes towards goblet

In [ ]:
scv.pl.velocity(adata_progen, var_names=['GALNT3', 'SELENOK', 'RAP1GAP'], colorbar=True, ncols=1, layers=['velocity', 'Ms'], use_raw=False, save='genesProgenGoblet', cmap='RdBu_r', color_map='Reds')

Stem velocity genes towards colonocytes

In [ ]:
scv.pl.velocity(adata_progen, var_names=['NR1H4', 'PDZD3', 'ENPP3'], colorbar=True, ncols=1, layers=['velocity', 'Ms'], use_raw=False, save='genesProgenColono', cmap='RdBu_r', color_map='Reds')

Stem velocity genes towards BEST4

In [ ]:
scv.pl.velocity(adata_progen, var_names=['CEACAM3', 'PROX1', 'CA7'], colorbar=True, ncols=1, layers=['velocity', 'Ms'], use_raw=False, save='genesProgenBEST4', cmap='RdBu_r', color_map='Reds')

In [ ]:
scv.tl.velocity_confidence(adata_progen)

In [ ]:
scv.pl.scatter(adata_progen, color='velocity_confidence', fontsize=24, size=100, colorbar=True, rescale_color=[0,1], color_map='RdBu_r')
scv.pl.scatter(adata_progen, color='velocity_confidence_transition', fontsize=24, size=100, colorbar=True, rescale_color=[0,1], color_map='RdBu_r')

Latent time

In [ ]:
scv.pl.scatter(adata_progen, color='latent_time', fontsize=24, size=100, title='', basis='umap',
               color_map='gnuplot', colorbar=True, rescale_color=[0,1], save='ProgenTime')

In [ ]:
scv.pl.scatter(adata_progen, color='root_cells')

## 8. GO analysis

In [ ]:
from goatools.base import download_go_basic_obo
obo_fname = download_go_basic_obo()

In [ ]:
from goatools.base import download_ncbi_associations
fin_gene2go = download_ncbi_associations()

In [ ]:
from goatools.obo_parser import GODag

obodag = GODag("go-basic.obo")

In [ ]:
from goatools.anno.genetogo_reader import Gene2GoReader

# Read NCBI's gene2go. Store annotations in a list of namedtuples
objanno = Gene2GoReader(fin_gene2go, taxids=[9606])

# Get namespace2association where:
#    namespace is:
#        BP: biological_process               
#        MF: molecular_function
#        CC: cellular_component
#    assocation is a dict:
#        key: NCBI GeneID
#        value: A set of GO IDs associated with that gene
ns2assoc = objanno.get_ns2assc()

for nspc, id2gos in ns2assoc.items():
    print("{NS} {N:,} annotated human genes".format(NS=nspc, N=len(id2gos)))

In [ ]:
# Use all genes as background genes
adata_qc
BackGroundGenes = adata_qc.var_names

In [ ]:
import mygene
mg = mygene.MyGeneInfo()

In [ ]:
BackGroundGenesEntrez = mg.querymany(BackGroundGenes, scopes='symbol', fields='entrezgene', species='human')

In [ ]:
# Extract just the entrez names that were found in the database (not NA)
BackGroundGenesEntrezValues = list(pd.DataFrame(BackGroundGenesEntrez)['entrezgene'].dropna().astype(int))

In [ ]:
print(len(BackGroundGenes))
print(len(BackGroundGenesEntrezValues))

In [ ]:
from goatools.goea.go_enrichment_ns import GOEnrichmentStudyNS

goeaobj = GOEnrichmentStudyNS(
        BackGroundGenesEntrezValues, # List of background human genes
        ns2assoc, # geneid/GO associations
        obodag, # Ontologies
        propagate_counts = False,
        alpha = 0.05, # default significance cut-off
        methods = ['fdr_bh']) # defult multipletest correction method

### 8.1 GO terms for crypt top populations 

In [ ]:
sc.tl.rank_genes_groups(adata, groupby='Celltype', method='wilcoxon', groups=['CT Goblet'], reference='rest', n_genes=10000)
deGobletCT = adata.uns['rank_genes_groups']

sc.tl.rank_genes_groups(adata, groupby='Celltype', method='wilcoxon', groups=['CT Colonocytes'], reference='rest', n_genes=10000)
deColonocytesCT = adata.uns['rank_genes_groups']

sc.tl.rank_genes_groups(adata, groupby='Celltype', method='wilcoxon', groups=['CT BEST4+'], reference='rest', n_genes=10000)
deBEST4CT = adata.uns['rank_genes_groups']

In [ ]:
deGobletCTdf = pd.DataFrame(data={'Name':deGobletCT['names'].astype(str),
                                  'Score':deGobletCT['scores'].astype(float),
                                  'log2FC':deGobletCT['logfoldchanges'].astype(float),
                                  'p-value':deGobletCT['pvals_adj'].astype(float)},
                            index=list(range(len(deGobletCT['names']))))
deGobletCTdf = deGobletCTdf[deGobletCTdf['log2FC']>0.5]
deGobletCTdf = deGobletCTdf[deGobletCTdf['p-value']<0.05]

In [ ]:
deColonocytesCTdf = pd.DataFrame(data={'Name':deColonocytesCT['names'].astype(str),
                                  'Score':deColonocytesCT['scores'].astype(float),
                                  'log2FC':deColonocytesCT['logfoldchanges'].astype(float),
                                  'p-value':deColonocytesCT['pvals_adj'].astype(float)},
                            index=list(range(len(deColonocytesCT['names']))))
deColonocytesCTdf = deColonocytesCTdf[deColonocytesCTdf['log2FC']>0.5]
deColonocytesCTdf = deColonocytesCTdf[deColonocytesCTdf['p-value']<0.05]

In [ ]:
deBEST4CTdf = pd.DataFrame(data={'Name':deBEST4CT['names'].astype(str),
                                  'Score':deBEST4CT['scores'].astype(float),
                                  'log2FC':deBEST4CT['logfoldchanges'].astype(float),
                                  'p-value':deBEST4CT['pvals_adj'].astype(float)},
                            index=list(range(len(deBEST4CT['names']))))
deBEST4CTdf = deBEST4CTdf[deBEST4CTdf['log2FC']>0.5]
deBEST4CTdf = deBEST4CTdf[deBEST4CTdf['p-value']<0.05]

In [ ]:
print(len(deGobletCTdf['Name']))
print(len(deColonocytesCTdf['Name']))
print(len(deBEST4CTdf['Name']))

In [ ]:
pd.DataFrame(adata.var_names).to_csv('all_genes.csv')

In [ ]:
deGobletCTdf['Name'].to_csv('DEGsGobletCT.csv')

In [ ]:
deColonocytesCTdf['Name'].to_csv('DEGsColonocytesCT.csv')

In [ ]:
deBEST4CTdf['Name'].to_csv('DEGsBEST4CT.csv')

In [ ]:
pd.DataFrame(CTgenesEntrezValues).to_csv('DEGsCommonCT.csv')

In [ ]:
pd.DataFrame(BackGroundGenesEntrezValues).to_csv('DEGsAll.csv')

In [ ]:
CTgenes = set(pd.concat([deGobletCTdf['Name'], deColonocytesCTdf['Name'], deBEST4CTdf['Name']]))

In [ ]:
len(set(CTgenes))

In [ ]:
commonCTgenes = list(set(set(deGobletCTdf['Name']) & set(deColonocytesCTdf['Name']) & set(deBEST4CTdf['Name'])))

In [ ]:
len(commonCTgenes)

In [ ]:
CTgenes = commonCTgenes

In [ ]:
CTgenes = set(deGobletCTdf['Name'])

In [ ]:
CTgenes = set(deColonocytesCTdf['Name'])

In [ ]:
CTgenes = set(deBEST4CTdf['Name'])

In [ ]:
data = "CT"

In [ ]:
# Turn gene symbols into Entrez IDs
CTgenesEntrez = mg.querymany(CTgenes, scopes='symbol', fields='entrezgene', species='human')
CTgenesEntrezValues = list(pd.DataFrame(CTgenesEntrez)['entrezgene'].dropna().astype(int))
CTgenesEntrezDict = dict(zip(list(pd.DataFrame(CTgenesEntrez)['entrezgene']), list(pd.DataFrame(CTgenesEntrez)['query'])))

In [ ]:
print(len(CTgenes))
print(len(CTgenesEntrezValues))

In [ ]:
# Run the GOEA
CT_goea_results_all = goeaobj.run_study(CTgenesEntrezValues)
CT_goea_results_sig = [r for r in CT_goea_results_all if r.p_fdr_bh < 0.05]

In [ ]:
goeaobj.wr_txt("{}goterms.txt".format(data), CT_goea_results_sig)
goeaobj.wr_xlsx("{}goterms.xlsx".format(data), CT_goea_results_sig)

In [ ]:
from goatools.godag_plot import plot_gos, plot_results, plot_goid2goobj

plot_results("allgoterms{NS}.pdf", CT_goea_results_sig)

In [ ]:
goid_subset = [
    'GO:0043312', # neutrophil degranulation
    'GO:0038096', # Fc-gamma receptor signaling pathway involved in phagocytosis
    'GO:0043123' # NF-kappaB
]
plot_gos("{}CTgoterms_immune.pdf".format(data),
    goid_subset, # Source GO ids
    obodag, 
    goea_results=CT_goea_results_sig) # Use pvals for coloring

In [ ]:
# change entrez into symbol
entrez = [387, 567, 634, 967, 978, 1476, 2495, 2517, 2934, 3728, 3958, 4680, 5646, 5660, 5879, 5906, 7879, 8649, 8655, 8673, 9218, 9798, 10487, 11240, 23593, 26578, 51316, 51382, 51646, 51719, 54509, 64114]
symbolsCT = mg.querymany(entrez, scopes='entrezgene', fields='symbol', species='human')
symbolsCT = list(pd.DataFrame(symbolsCT)['symbol'])
print(symbolsCT)

# STEM GO 

In [ ]:
StemInflamedDEgenesUP = pd.read_csv('StemGO/StemInflamedDEgenesUP.csv')

In [ ]:
StemInflamedDEgenesUP = list(StemInflamedDEgenesUP['Name'])

In [ ]:
StemNoninflamedDEgenesUP = pd.read_csv('StemGO/StemNoninflamedDEgenesUP.csv')

In [ ]:
StemNoninflamedDEgenesUP = list(StemNoninflamedDEgenesUP['Name'])

In [ ]:
StemCommonDEgenesUP = list(set(StemInflamedDEgenesUP) & set(StemNoninflamedDEgenesUP))

In [ ]:
len(StemCommonDEgenesUP)

In [ ]:
StemInflamedDEgenesDOWN = pd.read_csv('StemGO/StemInflamedDEgenesDOWN.csv')

In [ ]:
StemInflamedDEgenesDOWN = list(StemInflamedDEgenesDOWN['Name'])

In [ ]:
StemNoninflamedDEgenesDOWN = pd.read_csv('StemGO/StemNoninflamedDEgenesDOWN.csv')

In [ ]:
StemNoninflamedDEgenesDOWN = list(StemNoninflamedDEgenesDOWN['Name'])

In [ ]:
StemCommonDEgenesDOWN = list(set(StemInflamedDEgenesDOWN) & set(StemNoninflamedDEgenesDOWN))

In [ ]:
len(StemCommonDEgenesDOWN)

In [ ]:
data = "StemCommonDOWN"

In [ ]:
CTgenes = StemCommonDEgenesDOWN

In [ ]:
# Turn gene symbols into Entrez IDs
CTgenesEntrez = mg.querymany(CTgenes, scopes='symbol', fields='entrezgene', species='human')
CTgenesEntrezValues = list(pd.DataFrame(CTgenesEntrez)['entrezgene'].dropna().astype(int))
CTgenesEntrezDict = dict(zip(list(pd.DataFrame(CTgenesEntrez)['entrezgene']), list(pd.DataFrame(CTgenesEntrez)['query'])))

In [ ]:
print(len(CTgenes))
print(len(CTgenesEntrezValues))

In [ ]:
# Run the GOEA
CT_goea_results_all = goeaobj.run_study(CTgenesEntrezValues)
CT_goea_results_sig = [r for r in CT_goea_results_all if r.p_fdr_bh < 0.05]

In [ ]:
goeaobj.wr_txt("{}goterms.txt".format(data), CT_goea_results_sig)
goeaobj.wr_xlsx("{}goterms.xlsx".format(data), CT_goea_results_sig)

# LOAD DATA

In [ ]:
adata = sc.read('healthy_velo.h5ad')

### DE LAnge and extra

In [ ]:
with open('IBD_figures/UC.txt', 'r') as f:
    UC_genes = f.read().splitlines()
    
with open('IBD_figures/CD.txt', 'r') as f:
    CD_genes = f.read().splitlines()
    
with open('IBD_figures/IBD.txt', 'r') as f:
    IBD_genes = f.read().splitlines()

In [ ]:
UC_genes = list(set(UC_genes))
print(len(UC_genes))
UC_genes = [feature for feature in UC_genes if feature in adata.var_names]
print(len(UC_genes))

CD_genes = list(set(CD_genes))
print(len(CD_genes))
CD_genes = [feature for feature in CD_genes if feature in adata.var_names]
print(len(CD_genes))

IBD_genes = list(set(IBD_genes))
print(len(IBD_genes))
IBD_genes = [feature for feature in IBD_genes if feature in adata.var_names]
print(len(IBD_genes))

In [ ]:
# Input
from scipy.sparse import issparse
groupby = 'Celltype'
var_names = UC_genes

# Tidy data
matrix = adata.raw[:, var_names].X
if issparse(matrix):
    matrix = matrix.toarray()
obs_tidy = pd.DataFrame(matrix, columns=var_names)
categorical = adata.obs[groupby]
obs_tidy.set_index(categorical, groupby, inplace=True)
categories = obs_tidy.index.categories

obs_bool = obs_tidy > 0.0
fraction_obs_UC = obs_bool.groupby(level=0).sum() / obs_bool.groupby(level=0).count()

# Mean expression and scaled
mean_obs = obs_tidy.groupby(level=0).mean()
mean_obs -= mean_obs.min(0)
mean_obs_UC = (mean_obs / mean_obs.max(0)).fillna(0)

In [ ]:
# Input
from scipy.sparse import issparse
groupby = 'Celltype'
var_names = CD_genes

# Tidy data
matrix = adata.raw[:, var_names].X
if issparse(matrix):
    matrix = matrix.toarray()
obs_tidy = pd.DataFrame(matrix, columns=var_names)
categorical = adata.obs[groupby]
obs_tidy.set_index(categorical, groupby, inplace=True)
categories = obs_tidy.index.categories

obs_bool = obs_tidy > 0.0
fraction_obs_CD = obs_bool.groupby(level=0).sum() / obs_bool.groupby(level=0).count()

# Mean expression and scaled
mean_obs = obs_tidy.groupby(level=0).mean()
mean_obs -= mean_obs.min(0)
mean_obs_CD = (mean_obs / mean_obs.max(0)).fillna(0)

In [ ]:
# Input
from scipy.sparse import issparse
groupby = 'Celltype'
var_names = IBD_genes

# Tidy data
matrix = adata.raw[:, var_names].X
if issparse(matrix):
    matrix = matrix.toarray()
obs_tidy = pd.DataFrame(matrix, columns=var_names)
categorical = adata.obs[groupby]
obs_tidy.set_index(categorical, groupby, inplace=True)
categories = obs_tidy.index.categories

obs_bool = obs_tidy > 0.0
fraction_obs_IBD = obs_bool.groupby(level=0).sum() / obs_bool.groupby(level=0).count()

# Mean expression and scaled
mean_obs = obs_tidy.groupby(level=0).mean()
mean_obs -= mean_obs.min(0)
mean_obs_IBD = (mean_obs / mean_obs.max(0)).fillna(0)

In [ ]:
sc.tl.score_genes(adata, UC_genes, score_name='UC_genes')
sc.tl.score_genes(adata, CD_genes, score_name='CD_genes')
sc.tl.score_genes(adata, IBD_genes, score_name='IBD_genes')

In [ ]:
scv.set_figure_params(fontsize=12, vector_friendly=True, dpi=150, dpi_save=300, figsize=[4,4], format='png')
sc.pl.umap(adata, color='UC_genes', cmap='bwr', save="healthy-UC_genes")
sc.pl.umap(adata, color='CD_genes', cmap='bwr', save="healthy-CD_genes")
sc.pl.umap(adata, color='IBD_genes', cmap='bwr', save="healthy-IBD_genes")

In [ ]:
UC_fraction = pd.DataFrame(
        np.zeros((1,len(set(categorical)))),
        columns=set(categorical),
        index=['UC_fraction_in']
    )

In [ ]:
for celltype in set(categorical):
    n = 0
    for gene in UC_genes:
        if (fraction_obs_UC[gene][celltype] > 0):
            n += 1
    UC_fraction[celltype] = n

In [ ]:
UC_fraction

In [ ]:
UC_genes_expressed = []
for gene in UC_genes:
    if any (fraction_obs_UC[gene] >= 0.25):
        UC_genes_expressed.append(gene)

In [ ]:
CD_fraction = pd.DataFrame(
        np.zeros((1,len(set(categorical)))),
        columns=set(categorical),
        index=['CD_fraction_in']
    )

In [ ]:
for celltype in set(categorical):
    n = 0
    for gene in CD_genes:
        if (fraction_obs_CD[gene][celltype] > 0):
            n += 1
    CD_fraction[celltype] = n

In [ ]:
CD_fraction

In [ ]:
CD_genes_expressed = []
for gene in CD_genes:
    if any (fraction_obs_CD[gene] >= 0.25):
        CD_genes_expressed.append(gene)

In [ ]:
len(CD_genes_expressed)

In [ ]:
IBD_fraction = pd.DataFrame(
        np.zeros((1,len(set(categorical)))),
        columns=set(categorical),
        index=['IBD_fraction_in']
    )

In [ ]:
for celltype in set(categorical):
    n = 0
    for gene in IBD_genes:
        if (fraction_obs_IBD[gene][celltype] > 0):
            n += 1
    IBD_fraction[celltype] = n

In [ ]:
IBD_fraction

In [ ]:
IBD_genes_expressed = []
for gene in IBD_genes:
    if any (fraction_obs_IBD[gene] >= 0.25):
        IBD_genes_expressed.append(gene)

In [ ]:
len(IBD_genes_expressed)

## All genes

In [ ]:
adata.obs['all'] = adata.obs['Sample']
adata.obs['all'] = adata.obs['all'].replace("healthy0", "ALL")
adata.obs['all'] = adata.obs['all'].replace("healthy1", "ALL")
adata.obs['all'] = adata.obs['all'].replace("healthy7", "ALL")
adata.obs['all'] = adata.obs['all'].replace("healthy8", "ALL")

adata.obs['all'] = adata.obs['all'].astype('category')

In [ ]:
# Input
from scipy.sparse import issparse
groupby = 'all'
var_names = adata.var_names

# Tidy data
matrix = adata.raw[:, var_names].X
if issparse(matrix):
    matrix = matrix.toarray()
obs_tidy = pd.DataFrame(matrix, columns=var_names)
categorical = adata.obs[groupby]
obs_tidy.set_index(categorical, groupby, inplace=True)
categories = obs_tidy.index.categories

obs_bool = obs_tidy > 0.0
fraction_obs_all = obs_bool.groupby(level=0).sum() / obs_bool.groupby(level=0).count()

# Mean expression and scaled
mean_obs = obs_tidy.groupby(level=0).mean()
mean_obs -= mean_obs.min(0)
mean_obs_all = (mean_obs / mean_obs.max(0)).fillna(0)

In [ ]:
all_fraction = pd.DataFrame(
        np.zeros((1,len(set(categorical)))),
        columns=set(categorical),
        index=['all_fraction_in']
    )

In [ ]:
for celltype in set(categorical):
    n = 0
    for gene in adata.var_names:
        if (fraction_obs_all[gene][celltype] > 0):
            n += 1
    all_fraction[celltype] = n

In [ ]:
all_fraction

In [ ]:
all_fraction.to_csv('all_total_gene_fraction.csv')

In [ ]:
adata

## DE genes for GEA 

In [ ]:
sc.tl.rank_genes_groups(adata, groupby='Celltype', method='wilcoxon', groups=['Stem'], reference='rest', n_genes=10000)
deStem = adata.uns['rank_genes_groups']

sc.tl.rank_genes_groups(adata, groupby='Celltype', method='wilcoxon', groups=['TA SOX4+'], reference='rest', n_genes=10000)
deTASOX4 = adata.uns['rank_genes_groups']

sc.tl.rank_genes_groups(adata, groupby='Celltype', method='wilcoxon', groups=['TA Colonocytes'], reference='rest', n_genes=10000)
deTAColonocytes = adata.uns['rank_genes_groups']

sc.tl.rank_genes_groups(adata, groupby='Celltype', method='wilcoxon', groups=['Colonocytes'], reference='rest', n_genes=10000)
deColonocytes = adata.uns['rank_genes_groups']

sc.tl.rank_genes_groups(adata, groupby='Celltype', method='wilcoxon', groups=['CT Colonocytes'], reference='rest', n_genes=10000)
deCTColonocytes = adata.uns['rank_genes_groups']

sc.tl.rank_genes_groups(adata, groupby='Celltype', method='wilcoxon', groups=['TA BEST4+'], reference='rest', n_genes=10000)
deTABEST4 = adata.uns['rank_genes_groups']

sc.tl.rank_genes_groups(adata, groupby='Celltype', method='wilcoxon', groups=['BEST4+'], reference='rest', n_genes=10000)
deBEST4 = adata.uns['rank_genes_groups']

sc.tl.rank_genes_groups(adata, groupby='Celltype', method='wilcoxon', groups=['CT BEST4+'], reference='rest', n_genes=10000)
deCTBEST4 = adata.uns['rank_genes_groups']

sc.tl.rank_genes_groups(adata, groupby='Celltype', method='wilcoxon', groups=['TA Goblet'], reference='rest', n_genes=10000)
deTAGoblet = adata.uns['rank_genes_groups']

sc.tl.rank_genes_groups(adata, groupby='Celltype', method='wilcoxon', groups=['Goblet'], reference='rest', n_genes=10000)
deGoblet = adata.uns['rank_genes_groups']

sc.tl.rank_genes_groups(adata, groupby='Celltype', method='wilcoxon', groups=['CT Goblet'], reference='rest', n_genes=10000)
deCTGoblet = adata.uns['rank_genes_groups']

sc.tl.rank_genes_groups(adata, groupby='Celltype', method='wilcoxon', groups=['EECs'], reference='rest', n_genes=10000)
deEECs = adata.uns['rank_genes_groups']

sc.tl.rank_genes_groups(adata, groupby='Celltype', method='wilcoxon', groups=['Tuft'], reference='rest', n_genes=10000)
deTuft = adata.uns['rank_genes_groups']

In [ ]:
deStemdf = pd.DataFrame(data={'Name':deStem['names'].astype(str),
                                  'Score':deStem['scores'].astype(float),
                                  'log2FC':deStem['logfoldchanges'].astype(float),
                                  'p-value':deStem['pvals_adj'].astype(float)},
                            index=list(range(len(deStem['names']))))
deStemdf = deStemdf[deStemdf['log2FC']>0.5]
deStemdf = deStemdf[deStemdf['p-value']<0.05]

In [ ]:
deTASOX4df = pd.DataFrame(data={'Name':deTASOX4['names'].astype(str),
                                  'Score':deTASOX4['scores'].astype(float),
                                  'log2FC':deTASOX4['logfoldchanges'].astype(float),
                                  'p-value':deTASOX4['pvals_adj'].astype(float)},
                            index=list(range(len(deTASOX4['names']))))
deTASOX4df = deTASOX4df[deTASOX4df['log2FC']>0.5]
deTASOX4df = deTASOX4df[deTASOX4df['p-value']<0.05]

In [ ]:
deTAColonocytesdf = pd.DataFrame(data={'Name':deTAColonocytes['names'].astype(str),
                                  'Score':deTAColonocytes['scores'].astype(float),
                                  'log2FC':deTAColonocytes['logfoldchanges'].astype(float),
                                  'p-value':deTAColonocytes['pvals_adj'].astype(float)},
                            index=list(range(len(deTAColonocytes['names']))))
deTAColonocytesdf = deTAColonocytesdf[deTAColonocytesdf['log2FC']>0.5]
deTAColonocytesdf = deTAColonocytesdf[deTAColonocytesdf['p-value']<0.05]

In [ ]:
deColonocytesdf = pd.DataFrame(data={'Name':deColonocytes['names'].astype(str),
                                  'Score':deColonocytes['scores'].astype(float),
                                  'log2FC':deColonocytes['logfoldchanges'].astype(float),
                                  'p-value':deColonocytes['pvals_adj'].astype(float)},
                            index=list(range(len(deColonocytes['names']))))
deColonocytesdf = deColonocytesdf[deColonocytesdf['log2FC']>0.5]
deColonocytesdf = deColonocytesdf[deColonocytesdf['p-value']<0.05]

In [ ]:
deCTColonocytesdf = pd.DataFrame(data={'Name':deCTColonocytes['names'].astype(str),
                                  'Score':deCTColonocytes['scores'].astype(float),
                                  'log2FC':deCTColonocytes['logfoldchanges'].astype(float),
                                  'p-value':deCTColonocytes['pvals_adj'].astype(float)},
                            index=list(range(len(deCTColonocytes['names']))))
deCTColonocytesdf = deCTColonocytesdf[deCTColonocytesdf['log2FC']>0.5]
deCTColonocytesdf = deCTColonocytesdf[deCTColonocytesdf['p-value']<0.05]

In [ ]:
deTABEST4df = pd.DataFrame(data={'Name':deTABEST4['names'].astype(str),
                                  'Score':deTABEST4['scores'].astype(float),
                                  'log2FC':deTABEST4['logfoldchanges'].astype(float),
                                  'p-value':deTABEST4['pvals_adj'].astype(float)},
                            index=list(range(len(deTABEST4['names']))))
deTABEST4df = deTABEST4df[deTABEST4df['log2FC']>0.5]
deTABEST4df = deTABEST4df[deTABEST4df['p-value']<0.05]

In [ ]:
deBEST4df = pd.DataFrame(data={'Name':deBEST4['names'].astype(str),
                                  'Score':deBEST4['scores'].astype(float),
                                  'log2FC':deBEST4['logfoldchanges'].astype(float),
                                  'p-value':deBEST4['pvals_adj'].astype(float)},
                            index=list(range(len(deBEST4['names']))))
deBEST4df = deBEST4df[deBEST4df['log2FC']>0.5]
deBEST4df = deBEST4df[deBEST4df['p-value']<0.05]

In [ ]:
deCTBEST4df = pd.DataFrame(data={'Name':deCTBEST4['names'].astype(str),
                                  'Score':deCTBEST4['scores'].astype(float),
                                  'log2FC':deCTBEST4['logfoldchanges'].astype(float),
                                  'p-value':deCTBEST4['pvals_adj'].astype(float)},
                            index=list(range(len(deCTBEST4['names']))))
deCTBEST4df = deCTBEST4df[deCTBEST4df['log2FC']>0.5]
deCTBEST4df = deCTBEST4df[deCTBEST4df['p-value']<0.05]

In [ ]:
deTAGobletdf = pd.DataFrame(data={'Name':deTAGoblet['names'].astype(str),
                                  'Score':deTAGoblet['scores'].astype(float),
                                  'log2FC':deTAGoblet['logfoldchanges'].astype(float),
                                  'p-value':deTAGoblet['pvals_adj'].astype(float)},
                            index=list(range(len(deTAGoblet['names']))))
deTAGobletdf = deTAGobletdf[deTAGobletdf['log2FC']>0.5]
deTAGobletdf = deTAGobletdf[deTAGobletdf['p-value']<0.05]

In [ ]:
deGobletdf = pd.DataFrame(data={'Name':deGoblet['names'].astype(str),
                                  'Score':deGoblet['scores'].astype(float),
                                  'log2FC':deGoblet['logfoldchanges'].astype(float),
                                  'p-value':deGoblet['pvals_adj'].astype(float)},
                            index=list(range(len(deGoblet['names']))))
deGobletdf = deGobletdf[deGobletdf['log2FC']>0.5]
deGobletdf = deGobletdf[deGobletdf['p-value']<0.05]

In [ ]:
deCTGobletdf = pd.DataFrame(data={'Name':deCTGoblet['names'].astype(str),
                                  'Score':deCTGoblet['scores'].astype(float),
                                  'log2FC':deCTGoblet['logfoldchanges'].astype(float),
                                  'p-value':deCTGoblet['pvals_adj'].astype(float)},
                            index=list(range(len(deCTGoblet['names']))))
deCTGobletdf = deCTGobletdf[deCTGobletdf['log2FC']>0.5]
deCTGobletdf = deCTGobletdf[deCTGobletdf['p-value']<0.05]

In [ ]:
deEECsdf = pd.DataFrame(data={'Name':deEECs['names'].astype(str),
                                  'Score':deEECs['scores'].astype(float),
                                  'log2FC':deEECs['logfoldchanges'].astype(float),
                                  'p-value':deEECs['pvals_adj'].astype(float)},
                            index=list(range(len(deEECs['names']))))
deEECsdf = deEECsdf[deEECsdf['log2FC']>0.5]
deEECsdf = deEECsdf[deEECsdf['p-value']<0.05]

In [ ]:
deTuftdf = pd.DataFrame(data={'Name':deTuft['names'].astype(str),
                                  'Score':deTuft['scores'].astype(float),
                                  'log2FC':deTuft['logfoldchanges'].astype(float),
                                  'p-value':deTuft['pvals_adj'].astype(float)},
                            index=list(range(len(deTuft['names']))))
deTuftdf = deTuftdf[deTuftdf['log2FC']>0.5]
deTuftdf = deTuftdf[deTuftdf['p-value']<0.05]

In [ ]:
deStemdf['Name'].to_csv('DEGsStem.csv')
deTASOX4df['Name'].to_csv('DEGsTASOX4.csv')
deTAColonocytesdf['Name'].to_csv('DEGsTAColonocytes.csv')
deColonocytesdf['Name'].to_csv('DEGsColonocytes.csv')
deCTColonocytesdf['Name'].to_csv('DEGsCTColonocytes.csv')
deTABEST4df['Name'].to_csv('DEGsTABEST4.csv')
deBEST4df['Name'].to_csv('DEGsBEST4.csv')
deCTBEST4df['Name'].to_csv('DEGsCTBEST4.csv')
deTAGobletdf['Name'].to_csv('DEGsTAGoblet.csv')
deGobletdf['Name'].to_csv('DEGsGoblet.csv')
deCTGobletdf['Name'].to_csv('DEGsCTGoblet.csv')
deEECsdf['Name'].to_csv('DEGsEECs.csv')
deTuftdf['Name'].to_csv('DEGsTuft.csv')

### Total genes

In [ ]:
print("Stem:", len(deStemdf))
print("TA SOX4+:", len(deTASOX4df))
print("TA Colonocytes:", len(deTAColonocytesdf))
print("Colonocytes:", len(deColonocytesdf))
print("CT Colonocytes:", len(deCTColonocytesdf))
print("TA BEST4+:", len(deTABEST4df))
print("BEST4+:", len(deBEST4df))
print("CT BEST4+:", len(deCTBEST4df))
print("TA Goblet:", len(deTAGobletdf))
print("Goblet:", len(deGobletdf))
print("CT Goblet:", len(deCTGobletdf))
print("EECs:", len(deEECsdf))
print("Tuft:", len(deTuftdf))
print("Common CT:", len(commonCTgenes))

### UC genes

In [ ]:
print("Stem:", len([gene for gene in UC_genes if gene in list(deStemdf['Name'])]))
print("TA SOX4+:", len([gene for gene in UC_genes if gene in list(deTASOX4df['Name'])]))
print("TA Colonocytes:", len([gene for gene in UC_genes if gene in list(deTAColonocytesdf['Name'])]))
print("Colonocytes:", len([gene for gene in UC_genes if gene in list(deColonocytesdf['Name'])]))
print("CT Colonocytes:", len([gene for gene in UC_genes if gene in list(deCTColonocytesdf['Name'])]))
print("TA BEST4+:", len([gene for gene in UC_genes if gene in list(deTABEST4df['Name'])]))
print("BEST4+:", len([gene for gene in UC_genes if gene in list(deBEST4df['Name'])]))
print("CT BEST4+:", len([gene for gene in UC_genes if gene in list(deCTBEST4df['Name'])]))
print("TA Goblet:", len([gene for gene in UC_genes if gene in list(deTAGobletdf['Name'])]))
print("Goblet:", len([gene for gene in UC_genes if gene in list(deGobletdf['Name'])]))
print("CT Goblet:", len([gene for gene in UC_genes if gene in list(deCTGobletdf['Name'])]))
print("EECs:", len([gene for gene in UC_genes if gene in list(deEECsdf['Name'])]))
print("Tuft:", len([gene for gene in UC_genes if gene in list(deTuftdf['Name'])]))
print("Common CT:", len([gene for gene in UC_genes if gene in list(commonCTgenes)]))

### CD genes

In [ ]:
print("Stem:", len([gene for gene in CD_genes if gene in list(deStemdf['Name'])]))
print("TA SOX4+:", len([gene for gene in CD_genes if gene in list(deTASOX4df['Name'])]))
print("TA Colonocytes:", len([gene for gene in CD_genes if gene in list(deTAColonocytesdf['Name'])]))
print("Colonocytes:", len([gene for gene in CD_genes if gene in list(deColonocytesdf['Name'])]))
print("CT Colonocytes:", len([gene for gene in CD_genes if gene in list(deCTColonocytesdf['Name'])]))
print("TA BEST4+:", len([gene for gene in CD_genes if gene in list(deTABEST4df['Name'])]))
print("BEST4+:", len([gene for gene in CD_genes if gene in list(deBEST4df['Name'])]))
print("CT BEST4+:", len([gene for gene in CD_genes if gene in list(deCTBEST4df['Name'])]))
print("TA Goblet:", len([gene for gene in CD_genes if gene in list(deTAGobletdf['Name'])]))
print("Goblet:", len([gene for gene in CD_genes if gene in list(deGobletdf['Name'])]))
print("CT Goblet:", len([gene for gene in CD_genes if gene in list(deCTGobletdf['Name'])]))
print("EECs:", len([gene for gene in CD_genes if gene in list(deEECsdf['Name'])]))
print("Tuft:", len([gene for gene in CD_genes if gene in list(deTuftdf['Name'])]))
print("Common CT:", len([gene for gene in CD_genes if gene in list(commonCTgenes)]))

### IBD genes

In [ ]:
print("Stem:", len([gene for gene in IBD_genes if gene in list(deStemdf['Name'])]))
print("TA SOX4+:", len([gene for gene in IBD_genes if gene in list(deTASOX4df['Name'])]))
print("TA Colonocytes:", len([gene for gene in IBD_genes if gene in list(deTAColonocytesdf['Name'])]))
print("Colonocytes:", len([gene for gene in IBD_genes if gene in list(deColonocytesdf['Name'])]))
print("CT Colonocytes:", len([gene for gene in IBD_genes if gene in list(deCTColonocytesdf['Name'])]))
print("TA BEST4+:", len([gene for gene in IBD_genes if gene in list(deTABEST4df['Name'])]))
print("BEST4+:", len([gene for gene in IBD_genes if gene in list(deBEST4df['Name'])]))
print("CT BEST4+:", len([gene for gene in IBD_genes if gene in list(deCTBEST4df['Name'])]))
print("TA Goblet:", len([gene for gene in IBD_genes if gene in list(deTAGobletdf['Name'])]))
print("Goblet:", len([gene for gene in IBD_genes if gene in list(deGobletdf['Name'])]))
print("CT Goblet:", len([gene for gene in IBD_genes if gene in list(deCTGobletdf['Name'])]))
print("EECs:", len([gene for gene in IBD_genes if gene in list(deEECsdf['Name'])]))
print("Tuft:", len([gene for gene in IBD_genes if gene in list(deTuftdf['Name'])]))
print("Common CT:", len([gene for gene in IBD_genes if gene in list(commonCTgenes)]))

In [ ]:
adata

In [ ]:
ColonocytesALL = list(set(list(deTAColonocytesdf['Name']) + list(deColonocytesdf['Name']) + list(deCTColonocytesdf['Name'])))

In [ ]:
len(ColonocytesALL)

In [ ]:
GobletALL = list(set(list(deTAGobletdf['Name']) + list(deGobletdf['Name']) + list(deCTGobletdf['Name'])))

In [ ]:
len(GobletALL)

In [ ]:
BEST4ALL = list(set(list(deTABEST4df['Name']) + list(deBEST4df['Name']) + list(deCTBEST4df['Name'])))

In [ ]:
len(BEST4ALL)

UC genes

In [ ]:
print("Colonocytes:", len([gene for gene in UC_genes if gene in ColonocytesALL]))
print("BEST4:", len([gene for gene in UC_genes if gene in BEST4ALL]))
print("Goblet:", len([gene for gene in UC_genes if gene in GobletALL]))

CD genes

In [ ]:
print("Colonocytes:", len([gene for gene in CD_genes if gene in ColonocytesALL]))
print("BEST4:", len([gene for gene in CD_genes if gene in BEST4ALL]))
print("Goblet:", len([gene for gene in CD_genes if gene in GobletALL]))

IBD genes

In [ ]:
print("Colonocytes:", len([gene for gene in IBD_genes if gene in ColonocytesALL]))
print("BEST4:", len([gene for gene in IBD_genes if gene in BEST4ALL]))
print("Goblet:", len([gene for gene in IBD_genes if gene in GobletALL]))

# 9. Analysis of subset of EECs and Tuft

## 9.1 EECs

In [ ]:
EECs_names = adata1.obs_names[adata1.obs['Celltype']=='EECs']

In [ ]:
EECs_names_short = [i[:-9] for i in list(EECs_names.values)]

In [ ]:
adata_EECs = adata[adata.obs_names.isin(EECs_names_short)].copy()

In [ ]:
sc.pp.highly_variable_genes(adata_EECs, n_top_genes=2000, subset=False)

In [ ]:
corrected = sc.api.pp.mnn_correct(adata_EECs[adata_EECs.obs['Sample']=='healthy0'], adata_EECs[adata_EECs.obs['Sample']=='healthy1'], 
                                  adata_EECs[adata_EECs.obs['Sample']=='healthy7'], adata_EECs[adata_EECs.obs['Sample']=='healthy8'],
                                  batch_key='Sample', batch_categories=['healthy0', 'healthy1', 'healthy7', 'healthy8'],
                                  var_subset=list(adata_EECs.var_names[adata_EECs.var['highly_variable']]), k=15, var_adj=True, 
                                  do_concatenate=True, save_raw=True, n_jobs=12)

In [ ]:
adata_EECs = corrected[0].copy()

In [ ]:
sc.pp.highly_variable_genes(adata_EECs, n_top_genes=2000, subset=False)

In [ ]:
sc.tl.pca(adata_EECs, svd_solver='arpack')

In [ ]:
sc.external.pp.bbknn(adata_EECs, batch_key='Sample', neighbors_within_batch=5, n_pcs=30,
                     approx=False, use_faiss=False, metric='euclidean')

In [ ]:
sc.tl.umap(adata_EECs)

In [ ]:
sc.pl.umap(adata_EECs, color='Sample', size=800, save='_EECs_sample')

In [ ]:
sc.pl.umap(adata_EECs, color=['PYY', 'SCGN'], cmap='YlOrRd', size=800, save='_EECs_markers')

In [ ]:
sc.tl.leiden(adata_EECs, resolution=0.6)

In [ ]:
sc.set_figure_params(fontsize=12, vector_friendly=True, dpi=150, dpi_save=300)
sc.pl.umap(adata_EECs, color='leiden', legend_loc='right margin', title='', size=800, save='_EECs_cluster')

In [ ]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
sc.tl.rank_genes_groups(adata_EECs, groupby='leiden', method='wilcoxon')
sc.tl.filter_rank_genes_groups(adata_EECs, min_in_group_fraction=0.6, 
                               max_out_group_fraction=0.25, min_fold_change=1.5)
sc.pl.rank_genes_groups_dotplot(adata_EECs, n_genes=5, key='rank_genes_groups_filtered', 
                                standard_scale='var', color_map='bwr', dendrogram=False, save='_EECs_DEGs')

In [ ]:
sc.pl.umap(adata_EECs, color=['SLC38A11', 'INSL5'], cmap='YlOrRd', size=800, save='_EECs_clustermarkers')

In [ ]:
sc.pl.umap(adata_EECs, color=['SST', 'GCG', 'TIMP1', 'PYY'], 
           cmap='YlOrRd', size=800, ncols=2, use_raw=True, save='EECs_hormones')

In [ ]:
scv.pp.moments(adata_EECs, n_pcs=30, n_neighbors=20)
scv.tl.recover_dynamics(adata_EECs)

In [ ]:
scv.tl.velocity(adata_EECs, mode='dynamical', groupby='leiden')
scv.tl.velocity_graph(adata_EECs)#, n_neighbors=12)
scv.tl.recover_latent_time(adata_EECs)

In [ ]:
scv.set_figure_params(fontsize=12, vector_friendly=True, dpi=150, dpi_save=300, figsize=[4,4], format='png')

scv.pl.velocity_embedding_stream(adata_EECs, basis='umap', cmap='Reds',
                                 legend_fontsize=8, title='', 
                                 smooth=.7, min_mass=0.1, color='leiden',
                                 alpha=0.9, fontsize=30, legend_loc='none', size=800, save='_EECs_velocity')

In [ ]:
sc.set_figure_params(fontsize=12, vector_friendly=True, dpi=150, dpi_save=300)
sc.pl.umap(adata_EECs, color=['PYY', 'GCG', 'SST', 'TPH1', 'SCT', 'CCK', 'SCGN'], 
           cmap='YlOrRd', size=800, ncols=2, use_raw=True, save='_HEALTHY_EECs_NewMarkers')

In [ ]:
adata_EECs.write("healthy_EECs_velo.h5ad")

## 9.1 Tuft

In [ ]:
Tuft_names = adata1.obs_names[adata1.obs['Celltype']=='Tuft']

In [ ]:
Tuft_names_short = [i[:-9] for i in list(Tuft_names.values)]

In [ ]:
adata_Tuft = adata[adata.obs_names.isin(Tuft_names_short)].copy()

In [ ]:
sc.pp.highly_variable_genes(adata_Tuft, n_top_genes=2000, subset=False)

In [ ]:
corrected = sc.api.pp.mnn_correct(adata_Tuft[adata_Tuft.obs['Sample']=='healthy0'], adata_Tuft[adata_Tuft.obs['Sample']=='healthy1'], 
                                  adata_Tuft[adata_Tuft.obs['Sample']=='healthy7'], adata_Tuft[adata_Tuft.obs['Sample']=='healthy8'],
                                  batch_key='Sample', batch_categories=['healthy0', 'healthy1', 'healthy7', 'healthy8'],
                                  var_subset=list(adata_Tuft.var_names[adata_Tuft.var['highly_variable']]), k=15, var_adj=True, 
                                  do_concatenate=True, save_raw=True, n_jobs=12)

In [ ]:
adata_Tuft = corrected[0].copy()

In [ ]:
sc.pp.highly_variable_genes(adata_Tuft, n_top_genes=2000, subset=False)

In [ ]:
sc.tl.pca(adata_Tuft, svd_solver='arpack')

In [ ]:
sc.external.pp.bbknn(adata_Tuft, batch_key='Sample', neighbors_within_batch=5, n_pcs=30,
                     approx=False, use_faiss=False, metric='euclidean')

In [ ]:
sc.tl.umap(adata_Tuft)

In [ ]:
sc.pl.umap(adata_Tuft, color='Sample', size=600, save='_Tuft_sample')

In [ ]:
sc.pl.umap(adata_Tuft, color=['LRMP', 'SOX4'], cmap='YlOrRd', size=600, save='_Tuft_markers')

In [ ]:
sc.tl.leiden(adata_Tuft, resolution=0.6)

In [ ]:
sc.set_figure_params(fontsize=12, vector_friendly=True, dpi=150, dpi_save=300)
sc.pl.umap(adata_Tuft, color='leiden', legend_loc='right margin', title='', size=600, save='_Tuft_clusters')

In [ ]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
sc.tl.rank_genes_groups(adata_Tuft, groupby='leiden', method='wilcoxon')
sc.tl.filter_rank_genes_groups(adata_Tuft, min_in_group_fraction=0.6, 
                               max_out_group_fraction=0.25, min_fold_change=1.5)
sc.pl.rank_genes_groups_dotplot(adata_Tuft, n_genes=5, key='rank_genes_groups_filtered', 
                                standard_scale='var', color_map='bwr', dendrogram=False, save='_Tuft_DEGs')

In [ ]:
sc.pl.umap(adata_Tuft, color=['FABP5', 'CA2', 'NMU', 'VSNL1'], cmap='YlOrRd', size=600, save='_Tuft_clustermarkers')

In [ ]:
scv.pp.moments(adata_Tuft, n_pcs=30, n_neighbors=20)
scv.tl.recover_dynamics(adata_Tuft)

In [ ]:
scv.tl.velocity(adata_Tuft, mode='dynamical', groupby='leiden')
scv.tl.velocity_graph(adata_Tuft)#, n_neighbors=12)
scv.tl.recover_latent_time(adata_Tuft)

In [ ]:
scv.set_figure_params(fontsize=12, vector_friendly=True, dpi=150, dpi_save=300, figsize=[4,4], format='png')

scv.pl.velocity_embedding_stream(adata_Tuft, basis='umap', cmap='Reds',
                                 legend_fontsize=8, title='', 
                                 smooth=.5, min_mass=0.1, color='leiden',
                                 alpha=0.9, fontsize=30, legend_loc='none', size=600, save='_Tuft_velocity')

In [ ]:
adata_Tuft.write("healthy_Tuft_velo.h5ad")

## Inflammation signature  

In [ ]:
UC_inflammation_genes_UP = [x.strip() for x in open('data/UC_inflammation_genes_UP.txt')]

In [ ]:
len(UC_inflammation_genes_UP)

In [ ]:
UC_inflammation_genes_UP = [i for i in UC_inflammation_genes_UP if i in adata.var_names]

In [ ]:
len(UC_inflammation_genes_UP)

In [ ]:
sc.tl.score_genes(adata, UC_inflammation_genes_UP, score_name='UC_inflammation_genes_UP')

In [ ]:
sc.pl.umap(adata, color='UC_inflammation_genes_UP', cmap='bwr', save='UC_inflammation_genes_UP')

In [ ]:
UC_inflammation_genes_DOWN = [x.strip() for x in open('data/UC_inflammation_genes_DOWN.txt')]

In [ ]:
len(UC_inflammation_genes_DOWN)

In [ ]:
UC_inflammation_genes_DOWN = [i for i in UC_inflammation_genes_DOWN if i in adata.var_names]

In [ ]:
len(UC_inflammation_genes_DOWN)

In [ ]:
sc.tl.score_genes(adata, UC_inflammation_genes_DOWN, score_name='UC_inflammation_genes_DOWN')

In [ ]:
sc.pl.umap(adata, color='UC_inflammation_genes_DOWN', cmap='bwr', save='UC_inflammation_genes_DOWN')

In [ ]:
sc.pl.umap(adata, color='CD74', cmap='YlOrRd')

In [ ]:
adata.obs['CD74_expression'] = adata.raw[:,'CD74'].X.todense()

In [ ]:
CD74pos_names = adata.obs_names[adata.obs['CD74_expression'] > 0.3970]

In [ ]:
CD74neg_names = adata.obs_names[adata.obs['CD74_expression'] <= 0.3970]

In [ ]:
len(CD74pos_names)*100/len(adata.obs_names)

In [ ]:
len(CD74neg_names)*100/len(adata.obs_names)

In [ ]:
CD74pos_names = adata.obs_names[adata.obs['CD74_expression'] > 1.3]

In [ ]:
CD74neg_names = adata.obs_names[adata.obs['CD74_expression'] <= 1.3]

In [ ]:
len(CD74pos_names)*100/len(adata.obs_names)

In [ ]:
len(CD74neg_names)*100/len(adata.obs_names)

In [ ]:
sc.pl.umap(adata, color='POLR1A', cmap='YlOrRd')